# 09 — Phòng thủ đầu-cuối với truy hồi thật, và quyền báo "ngữ cảnh mâu thuẫn" (bổ sung kế hoạch v1.5)

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

Notebook 05–08 đặt ngữ cảnh **bằng tay** (oracle): mình chọn sẵn đoạn cũ, đoạn mới và đoạn nhiễu. Phản biện hỏi hai câu mà các notebook đó chưa trả lời:

1. **Trong một hệ RAG thật** (bộ truy hồi tự chọn đoạn), lỗi thời có xảy ra không, và **cách phòng thủ nào** thật sự hiệu quả?
2. Khi ngữ cảnh không gắn nhãn chứa hai phiên bản mâu thuẫn, mô hình **có nhận ra mâu thuẫn** không, nếu được phép báo?

Notebook này trả lời cả hai bằng hai thí nghiệm, trên **cả 6 mô hình** và **cả 2 nhánh**:

| | Nội dung | Cách kết luận |
|---|---|---|
| **E4** | Truy hồi BM25 top-4 trên **kho trộn** hai phiên bản (hệ thống nạp thêm bản mới mà không xoá bản cũ), so 7 cấu hình: từ không có RAG, kho trộn ngây thơ, tới phòng thủ trọn bộ | 3 giả thuyết R1–R3, kiểm định một phía, Holm trong từng mô hình; **được ủng hộ** nếu đạt ở ≥ ⌈0,8 × số mô hình đánh giá được⌉ và không mô hình nào ngược chiều |
| **E5** | Ngữ cảnh oracle không nhãn (O,N và N,O), có và không có trường `"conflict"` trong lược đồ JSON | chỉ mô tả: tỉ lệ báo mâu thuẫn, hiệu và KTC 95% hai phía |

Ngoài ra notebook đo **độ lặp lại**: prompt `old_new`, `new_old` trùng từng byte với notebook 07, nên so được câu trả lời của lần chạy này với lần trước.

**Đây là phần mở rộng, không phải thí nghiệm khẳng định.** Kết luận của notebook 07 (v1.3) và 08 (v1.4) giữ nguyên. Kế hoạch cho phần này (bổ sung v1.5) được viết **trước khi** có bất kỳ câu trả lời nào của notebook này.

**Cách chạy:** giống notebook 07–08. Chạy **hai lần**: lần đầu `TRANSITION = "18.1-19.2"`, lần sau `TRANSITION = "6.3-7.2"`. Kết quả ghi vào `results/e2e-<lần chuyển>/`. Mỗi lần chạy độc lập, không có mục nào cần kết quả của cả hai.

**Điều kiện trước khi chạy:** các thư mục kết quả cũ phải còn nguyên:

- `results/conf-<lần chuyển>/` (notebook 07): để kiểm tra prompt trùng từng byte, lấy cấu hình môi trường và so độ lặp lại;
- `results/ext14b-<lần chuyển>/` và `results/ext14b_revision.json` (notebook 08): mã commit và cấu hình của Qwen3-14B;
- nhánh A: `results/mech-18.1-19.2/` (notebook 06), để so độ lặp lại cho Qwen3-8B và Llama-3.1-8B.

**Thời gian ước tính** trên RTX 4070: mỗi nhánh khoảng 2,5–3,5 giờ (5 mô hình 8B mỗi mô hình 15–30 phút, Qwen3-14B 45–75 phút). Cả hai nhánh khoảng 5–7 giờ.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.5-e2e | 14/10/2026 | Bản đầu tiên của notebook 09, theo bổ sung kế hoạch v1.5. Câu hỏi, lời dặn hệ thống, bộ chấm, phép kiểm định và module suy luận lấy nguyên từ notebook 07–08. |

## 1. Cấu hình

**Em chỉ cần đổi `TRANSITION`.** Mọi tham số khác đã cố định theo bổ sung kế hoạch v1.5.

**Mô hình:** đúng 6 mô hình, đúng mã commit của notebook 07–08 (Qwen3-14B đọc mã đã ghim ở `results/ext14b_revision.json`). Mô hình đã có sẵn trong `/data/hf`, không cần tải lại.

**Cấu hình môi trường:** mỗi mô hình dùng **đúng thang dự phòng** đã dùng ở notebook trước:

| Mô hình | Thang | Cấu hình bắt đầu |
|---|---|---|
| 5 mô hình 8B | thang của notebook 07: mặc định (`gpu_mem` 0,85, `max_model_len` 6144, Phi-4-mini 4096) → cuBLASLt → gọn bộ nhớ → cả hai | cấu hình đã dùng ở notebook 07 cùng nhánh (không có thì 0) |
| Qwen3-14B | thang của notebook 08: gọn bộ nhớ (0,92; 3072) → + cuBLASLt → rất gọn bộ nhớ (0,95; 2560) → + cuBLASLt | cấu hình đã dùng ở notebook 08 cùng nhánh (lần trước là 2) |

Chỉ đi lên thang, không quay xuống. Nhánh B không bắt đầu thấp hơn cấu hình mô hình đó đã dùng ở nhánh A của notebook này.

**Trước khi chạy:** tắt mọi chương trình khác đang dùng GPU (kiểm tra bằng `nvidia-smi`).

In [1]:
TRANSITION = "6.3-7.2"        # ĐỔI Ở ĐÂY: "18.1-19.2" (nhánh A) hoặc "6.3-7.2" (nhánh B)

from pathlib import Path
import os, sys, json, re, math, time, hashlib, subprocess, collections, datetime, zipfile, signal, random
import numpy as np
import pandas as pd
from IPython.display import display

V_OLD, V_NEW = TRANSITION.split("-")
ROOT    = Path.cwd()
DRIFT   = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
OUT     = ROOT / "results" / f"e2e-{V_OLD}-{V_NEW}"
CONF07  = ROOT / "results" / f"conf-{V_OLD}-{V_NEW}"      # notebook 07 (prompt, cấu hình, độ lặp lại)
EXT08   = ROOT / "results" / f"ext14b-{V_OLD}-{V_NEW}"    # notebook 08 (Qwen3-14B)
PROMPTS, PROMPTS_MS = OUT / "prompts", OUT / "prompts_mergedsys"
SMOKE_P, SMOKE_P_MS = OUT / "prompts_smoke", OUT / "prompts_smoke_mergedsys"
RAW, LOGS = OUT / "raw", OUT / "logs"
for p in (OUT, PROMPTS, PROMPTS_MS, SMOKE_P, SMOKE_P_MS, RAW, LOGS):
    p.mkdir(parents=True, exist_ok=True)

HF_HOME         = "/data/hf"
VLLM_CACHE_ROOT = "/data/vllm-cache"
REV_FILE = ROOT / "results" / "ext14b_revision.json"
assert REV_FILE.exists(), "Thiếu results/ext14b_revision.json của notebook 08 — cần giữ nguyên kết quả notebook 08"
_rev = json.loads(REV_FILE.read_text())
assert _rev["repo"] == "Qwen/Qwen3-14B-AWQ" and re.fullmatch(r"[0-9a-f]{40}", _rev["revision"]), REV_FILE
MODELS = [   # thứ tự chạy: 5 mô hình 8B trước, Qwen3-14B (chậm nhất) sau cùng
    dict(tag="qwen3-8b", repo="Qwen/Qwen3-8B-AWQ", revision="4da05a8edb55c6046cce958586c33b61da07bb79",
         chat_template_kwargs={"enable_thinking": False}, merge_system=False, ladder="8b"),
    dict(tag="llama3.1-8b", repo="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
         revision="db1f81ad4b8c7e39777509fac66c652eb0a52f91", chat_template_kwargs={}, merge_system=False, ladder="8b"),
    dict(tag="gemma2-9b", repo="hugging-quants/gemma-2-9b-it-AWQ-INT4",
         revision="6e62725da8e92309167814dad7aacc0ed8cb2484", chat_template_kwargs={}, merge_system=True, ladder="8b"),
    dict(tag="phi4-mini", repo="microsoft/Phi-4-mini-instruct",
         revision="cfbefacb99257ffa30c83adab238a50856ac3083", chat_template_kwargs={}, merge_system=False, ladder="8b",
         max_model_len=4096),   # như notebook 07: ≤ original_max_position_embeddings
    dict(tag="mistral7b-v0.3", repo="solidrust/Mistral-7B-Instruct-v0.3-AWQ",
         revision="95b1295ddd1a8673117cdc7bd2a4da2a457bb3f7", chat_template_kwargs={}, merge_system=True, ladder="8b"),
    dict(tag="qwen3-14b", repo="Qwen/Qwen3-14B-AWQ", revision=_rev["revision"],
         chat_template_kwargs={"enable_thinking": False}, merge_system=False, ladder="14b"),
]

STUDY = {
    "18.1-19.2": dict(branch="A", source="pilot_queries.jsonl",
                      templates=["D1_tech_tactic", "D2_subtech_parent", "D4_id_currency"], stable_cap=None),
    "6.3-7.2":   dict(branch="B", source="queries_all.jsonl",
                      templates=["D1_tech_tactic", "D4_id_currency"], stable_cap=80),
}
assert TRANSITION in STUDY, f"TRANSITION phải là một trong {list(STUDY)}"
CFG = STUDY[TRANSITION]

# kind: closed (không RAG) | rag (truy hồi BM25 trên kho trộn) | oracle (ngữ cảnh đặt sẵn như notebook 05–08)
CONDITIONS = {
    "closed":               dict(kind="closed"),
    "mixed":                dict(kind="rag", pool="mixed",   query="question", label=False, order="rank",          note=False),
    "mixed_label":          dict(kind="rag", pool="mixed",   query="question", label=True,  order="rank",          note=False),
    "mixed_label_curfirst": dict(kind="rag", pool="mixed",   query="question", label=True,  order="current_first", note=False),
    "filter":               dict(kind="rag", pool="current", query="question", label=False, order="rank",          note=False),
    "filter_exp":           dict(kind="rag", pool="current", query="expanded", label=False, order="rank",          note=False),
    "filter_exp_note":      dict(kind="rag", pool="current", query="expanded", label=False, order="rank",          note=True),
    "old_new":              dict(kind="oracle", order=["O", "N"], conflict=False),
    "new_old":              dict(kind="oracle", order=["N", "O"], conflict=False),
    "cf_old_new":           dict(kind="oracle", order=["O", "N"], conflict=True),
    "cf_new_old":           dict(kind="oracle", order=["N", "O"], conflict=True),
}
RAG_CONDS = [c for c, s in CONDITIONS.items() if s["kind"] == "rag"]
TOP_K           = 4          # số đoạn truy hồi (bằng số đoạn ở ngữ cảnh oracle; cũng là mặc định k = 4 của LangChain)
N_PASSAGES      = 4          # ngữ cảnh oracle: như notebook 05–08
MAX_CHUNK_WORDS = 250
MAX_TOKENS      = 160
SEED            = 2026
BOOT_B          = 10000
ALPHA           = 0.05
FLOOR           = 0.15       # R1, R3: lỗi thời ở `mixed` dưới mức này -> không đánh giá được
CEIL            = 0.85       # R2: độ chính xác ở `filter` (mã bị thu hồi) trên mức này -> không đánh giá được
PASS_SHARE      = 0.8        # tỉ lệ mô hình phải đạt
JSON_MIN        = 0.95
SMOKE_PER_GROUP = 1
CONF_VERSION    = "1.5-e2e"

# Thang cấu hình môi trường (định trước): thang của notebook 07 cho mô hình 8B, của notebook 08 cho Qwen3-14B
LEAN  = {"enforce_eager": True, "max_num_seqs": 32, "max_num_batched_tokens": 2048}
LEAN2 = {"enforce_eager": True, "max_num_seqs": 8, "max_num_batched_tokens": 1024}
LT    = {"TORCH_BLAS_PREFER_CUBLASLT": "1"}
LADDERS = {
    "8b": [
        dict(name="mặc định",                  env={}, engine={},    gpu_mem=0.85, max_model_len=6144),
        dict(name="cuBLASLt",                  env=LT, engine={},    gpu_mem=0.85, max_model_len=6144),
        dict(name="gọn bộ nhớ",                env={}, engine=LEAN,  gpu_mem=0.80, max_model_len=4096),
        dict(name="cuBLASLt + gọn bộ nhớ",     env=LT, engine=LEAN,  gpu_mem=0.80, max_model_len=4096),
    ],
    "14b": [
        dict(name="gọn bộ nhớ",                env={}, engine=LEAN,  gpu_mem=0.92, max_model_len=3072),
        dict(name="gọn bộ nhớ + cuBLASLt",     env=LT, engine=LEAN,  gpu_mem=0.92, max_model_len=3072),
        dict(name="rất gọn bộ nhớ",            env={}, engine=LEAN2, gpu_mem=0.95, max_model_len=2560),
        dict(name="rất gọn bộ nhớ + cuBLASLt", env=LT, engine=LEAN2, gpu_mem=0.95, max_model_len=2560),
    ],
}

REQUIRED = {
    "queries":        DRIFT / CFG["source"],
    "items":          DRIFT / "items.jsonl",
    "corpus_mixed":   DRIFT / "corpus_mixed.jsonl",
    "drift_manifest": DRIFT / "manifest.json",
    "tactic_alias":   DRIFT / "tactic_alias.json",
    "corpus_old":     ROOT / "benchmark" / f"attack-{V_OLD}" / "corpus.jsonl",
    "corpus_new":     ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
    "scoring_module": ROOT / "ctidrift_scoring.py",
    "scoring_v13":    ROOT / "ctidrift_scoring_v13.py",
    "manifest_07":    CONF07 / "manifest.json",
    "manifest_08":    EXT08 / "manifest.json",
}
missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, "Thiếu đầu vào (cần giữ nguyên kết quả notebook 02, 03, 07, 08):\n  " + "\n  ".join(missing)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

DM = json.loads(REQUIRED["drift_manifest"].read_text())
for k, f in (("queries", CFG["source"]), ("items", "items.jsonl"), ("corpus_mixed", "corpus_mixed.jsonl")):
    assert DM["outputs"][f] == sha256(REQUIRED[k]), f"{f} không khớp manifest notebook 03"
M07 = json.loads(REQUIRED["manifest_07"].read_text())
M08 = json.loads(REQUIRED["manifest_08"].read_text())
for k in ("queries", "items", "corpus_old", "corpus_new", "tactic_alias", "scoring_module", "scoring_v13"):
    assert M07["inputs"][k]["sha256"] == sha256(REQUIRED[k]), f"{k} khác đầu vào của notebook 07 — gửi thầy"
for m in MODELS:   # cùng mã commit với notebook 07/08
    if m["tag"] == "qwen3-14b":
        assert any(x["tag"] == "qwen3-14b" and x["revision"] == m["revision"] for x in M08["models"]), "mã commit 14B lệch"

ENV = {**os.environ, "HF_HOME": HF_HOME, "HF_HUB_CACHE": f"{HF_HOME}/hub", "VLLM_CACHE_ROOT": VLLM_CACHE_ROOT,
       "VLLM_USE_FLASHINFER_SAMPLER": "0", "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("HF_HUB_OFFLINE", None)
ENV.pop("TORCH_BLAS_PREFER_CUBLASLT", None)     # cấu hình BLAS chỉ do thang dự phòng quyết định
ENV.pop("TORCH_BLAS_PREFER_HIPBLASLT", None)

MANIFEST = {"conf_version": CONF_VERSION, "transition": TRANSITION, "branch": CFG["branch"],
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(CONDITIONS=CONDITIONS, STUDY=STUDY, TOP_K=TOP_K, N_PASSAGES=N_PASSAGES,
                           MAX_CHUNK_WORDS=MAX_CHUNK_WORDS, MAX_TOKENS=MAX_TOKENS, SEED=SEED, BOOT_B=BOOT_B,
                           ALPHA=ALPHA, FLOOR=FLOOR, CEIL=CEIL, PASS_SHARE=PASS_SHARE, JSON_MIN=JSON_MIN,
                           LADDERS=LADDERS),
            "models": MODELS, "plan": "bổ sung kế hoạch v1.5 (phòng thủ đầu-cuối, quyền báo mâu thuẫn)",
            "inputs": {k: {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)} for k, p in REQUIRED.items()}}
print(f"Notebook 09, nhánh {CFG['branch']}: ATT&CK v{V_OLD} → v{V_NEW}; mô hình: {[m['tag'] for m in MODELS]}")
for k, v in MANIFEST["inputs"].items():
    print(f"  {k:15s} {v['sha256'][:12]}…  {v['path']}")

Notebook 09, nhánh B: ATT&CK v6.3 → v7.2; mô hình: ['qwen3-8b', 'llama3.1-8b', 'gemma2-9b', 'phi4-mini', 'mistral7b-v0.3', 'qwen3-14b']
  queries         58a6ccb4646c…  benchmark/drift-6.3-7.2/queries_all.jsonl
  items           c4849c308e3a…  benchmark/drift-6.3-7.2/items.jsonl
  corpus_mixed    d1013db813ea…  benchmark/drift-6.3-7.2/corpus_mixed.jsonl
  drift_manifest  d5ada2826ad4…  benchmark/drift-6.3-7.2/manifest.json
  tactic_alias    e082cfab19e8…  benchmark/drift-6.3-7.2/tactic_alias.json
  corpus_old      127261d010d2…  benchmark/attack-6.3/corpus.jsonl
  corpus_new      1062f267ff29…  benchmark/attack-7.2/corpus.jsonl
  scoring_module  56b3cb0eb0c8…  ctidrift_scoring.py
  scoring_v13     0353476caf82…  ctidrift_scoring_v13.py
  manifest_07     f914d8b17e51…  results/conf-6.3-7.2/manifest.json
  manifest_08     a0af961a7a66…  results/ext14b-6.3-7.2/manifest.json


## 2. Tập câu hỏi và bản đồ dòng dõi

**Câu hỏi:** đúng tập câu của notebook 07–08 cùng nhánh, chỉ cách hỏi `ver` (câu hỏi có số phiên bản mới):

- nhánh A: các câu pilot, mẫu D1, D2, D4, ba nhóm `changed`, `label`, `stable`;
- nhánh B: mọi câu `changed` của D1 và D4, cộng 80 câu `stable` mỗi mẫu (hạt giống cố định).

**Bản đồ dòng dõi** (dùng cho phòng thủ): mọi quan hệ `revoked-by` của bản mới, lấy từ `items.jsonl` của notebook 03 (notebook 03 đã đối chiếu chúng với STIX và release notes). Một hệ thống thật đọc được đúng thông tin này từ bản STIX hiện hành. Bản đồ cho biết: mã cũ nào bị thu hồi, tên cũ của nó, và (các) mã kế nhiệm kèm tên.

In [2]:
import importlib, ctidrift_scoring, ctidrift_scoring_v13
importlib.reload(ctidrift_scoring); importlib.reload(ctidrift_scoring_v13)
from ctidrift_scoring import score_drift, _json_objects
from ctidrift_scoring_v13 import score_v13

ALIAS_FILE = json.loads(REQUIRED["tactic_alias"].read_text())
TACTIC_ALIAS = ALIAS_FILE["alias"]
ITEMS = {it["item_id"]: it for it in map(json.loads, open(REQUIRED["items"], encoding="utf-8"))}
CORP = {V_OLD: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_old"], encoding="utf-8"))},
        V_NEW: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_new"], encoding="utf-8"))}}

def group_of(q):                         # giống notebook 05–08
    if q["answer_changed"]:
        return "changed"
    if q["pair_class"] == "LABEL_CHANGED":
        return "label"
    if q["pair_class"] == "STABLE":
        return "stable"
    return None

def evidence_new(q, it):                 # giống notebook 05–08
    ev = q["evidence"][V_NEW]
    return ev if ev else [f"{s}#0" for s in it["anchor_new"] if f"{s}#0" in CORP[V_NEW]]

cand = []
for q in map(json.loads, open(REQUIRED["queries"], encoding="utf-8")):
    g = group_of(q)
    if q["direction"] != "current" or q["framing"] != "current" or q["template"] not in CFG["templates"] or g is None:
        continue
    cand.append({**q, "group": g})
if CFG["stable_cap"]:
    keep = set()
    for tpl in CFG["templates"]:
        pool = sorted(q["query_id"] for q in cand if q["template"] == tpl and q["group"] == "stable")
        keep |= set(random.Random(f"{SEED}-{tpl}-stable").sample(pool, min(CFG["stable_cap"], len(pool))))
    cand = [q for q in cand if q["group"] != "stable" or q["query_id"] in keep]

QUERIES, EV = [], {}
for q in cand:
    it = ITEMS[q["item_id"]]
    eo, en = q["evidence"][V_OLD], evidence_new(q, it)
    assert eo and en, f"{q['query_id']}: thiếu bằng chứng O hoặc N"
    assert all(e in CORP[V_OLD] for e in eo) and all(e in CORP[V_NEW] for e in en)
    QUERIES.append({**q, "base_id": q["query_id"], "qform": "ver"})
    EV[q["query_id"]] = (eo, en)
QBYID = {q["query_id"]: q for q in QUERIES}
REVOKED = {q["query_id"]: ITEMS[q["item_id"]]["anchor_status_new"] != "active" for q in QUERIES}

# Cùng tập câu hỏi với notebook 07 (nhóm ver)
q07 = {json.loads(l)["query_id"] for l in open(CONF07 / "queries.jsonl", encoding="utf-8")}
assert {q["query_id"] for q in QUERIES} == {k for k in q07 if not k.endswith("_nover")}, "Tập câu hỏi khác notebook 07"
with open(OUT / "queries.jsonl", "w", encoding="utf-8") as f:
    for q in QUERIES:
        f.write(json.dumps(q, ensure_ascii=False, sort_keys=True) + "\n")

# Bản đồ dòng dõi: mã bị thu hồi -> tên cũ, [(mã kế nhiệm, tên)]
LINEAGE = {}
for it in ITEMS.values():
    if not it["anchor_old"] or it["anchor_status_new"] == "active":
        continue
    assert it["anchor_status_new"] == "revoked" and it["anchor_new"], f"{it['item_id']}: trạng thái lạ"
    names = it["name_new"].split(" / ") if len(it["anchor_new"]) > 1 else [it["name_new"]]
    assert len(names) == len(it["anchor_new"]), f"{it['item_id']}: số tên và số mã kế nhiệm không khớp"
    v = dict(name_old=it["name_old"], succ=[list(x) for x in zip(it["anchor_new"], names)])
    assert LINEAGE.setdefault(it["anchor_old"], v) == v, f"{it['anchor_old']}: dòng dõi không nhất quán giữa các mục"
_byname = collections.defaultdict(set)
for k, v in LINEAGE.items():
    _byname[v["name_old"]].add(k)
NAME2OLD = {n: next(iter(s)) for n, s in _byname.items() if len(s) == 1}   # tên mơ hồ (nhiều mã) thì không dùng
print(f"{len(QUERIES)} câu hỏi:", dict(collections.Counter((q['template'][:2], q['group']) for q in QUERIES)))
print("Câu changed có mã bị thu hồi:", dict(collections.Counter(q["template"][:2] for q in QUERIES
                                                             if q["group"] == "changed" and REVOKED[q["query_id"]])))
print(f"Bản đồ dòng dõi: {len(LINEAGE)} mã bị thu hồi; {len(_byname) - len(NAME2OLD)} tên cũ mơ hồ (bỏ qua)")

361 câu hỏi: {('D1', 'stable'): 80, ('D1', 'changed'): 72, ('D4', 'stable'): 80, ('D4', 'changed'): 129}
Câu changed có mã bị thu hồi: {'D1': 62, 'D4': 129}
Bản đồ dòng dõi: 129 mã bị thu hồi; 0 tên cũ mơ hồ (bỏ qua)


## 3. Ngữ cảnh oracle cho E5 (giống notebook 05–08)

Đoạn nhiễu và cách dựng ngữ cảnh lấy **nguyên văn** từ notebook 07–08. Ô kiểm tra ở mục 5 xác nhận prompt `old_new`, `new_old` trùng từng byte với notebook 07. Ghi chú cầu nối (`BRIDGE`) của notebook 07 chỉ dùng để **đối chiếu** với ghi chú dòng dõi ở mục 4.

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class BM25:                            # giống hệt notebook 04–08
    def __init__(self, docs, k1=1.2, b=0.75):
        self.ids = sorted(docs)
        n = len(self.ids)
        toks = [collections.Counter(tokenize(docs[i])) for i in self.ids]
        dl = np.array([sum(t.values()) for t in toks], dtype=float)
        avgdl = dl.mean()
        post = collections.defaultdict(lambda: ([], []))
        for j, tf in enumerate(toks):
            for t, c in tf.items():
                post[t][0].append(j)
                post[t][1].append(c)
        self.post = {}
        for t, (idx, tf) in post.items():
            idx, tf = np.array(idx), np.array(tf, dtype=float)
            idf = math.log(1 + (n - len(idx) + 0.5) / (len(idx) + 0.5))
            self.post[t] = (idx, idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[idx] / avgdl)))
        self.n = n

    def search(self, query, k=5):
        scores = np.zeros(self.n)
        for t in sorted(set(tokenize(query))):
            if t in self.post:
                idx, w = self.post[t]
                scores[idx] += w
        cand = np.nonzero(scores)[0]
        order = cand[np.lexsort((cand, -scores[cand]))][:k]
        return [(self.ids[j], float(scores[j])) for j in order]

INDEX_NEW = BM25({i: d["title"] + "\n" + d["text"] for i, d in CORP[V_NEW].items()})
_old_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["new_only_labels"]}
_new_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["old_only_labels"]}
OLD_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_old_names)}
NEW_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_new_names)}

def bridge_note(q):                    # giống hệt notebook 07–08 (chỉ để đối chiếu)
    it = ITEMS[q["item_id"]]
    if q["group"] != "changed" or it["anchor_status_new"] == "active":
        return None
    major = V_NEW.split(".")[0]
    names = it["name_new"].split(" / ") if len(it["anchor_new"]) > 1 else [it["name_new"]]
    assert len(names) == len(it["anchor_new"]), f"{it['item_id']}: số tên và số mã kế nhiệm không khớp"
    repl = ", ".join(f"{s} ({n})" for s, n in zip(it["anchor_new"], names))
    return (f"In MITRE ATT&CK v{major}, technique {it['anchor_old']} ({it['name_old']}) was revoked "
            f"and replaced by {repl}.")

def banned_terms(q):
    it = ITEMS[q["item_id"]]
    ids = {it["anchor_old"], *it["anchor_new"], *q["gold"], *q["other_gold"]}
    names = set()
    if q["answer_type"] == "tactic_set":
        for t in set(q["gold"]) | set(q["other_gold"]):
            names |= {OLD_NAME.get(t), NEW_NAME.get(t)} - {None}
    return ids, names

def mentions_answer(text, ids, names):
    low = text.lower()
    return any(re.search(re.escape(i) + r"(?![.\d])", text) for i in ids) or any(n.lower() in low for n in names)

def distractors(q, k):
    ids, names = banned_terms(q)
    eo, en = EV[q["query_id"]]
    out = []
    for cid, _ in INDEX_NEW.search(q["question"], 3000):
        d = CORP[V_NEW][cid]
        if cid in en or d["metadata"]["doc_id"] in ids or mentions_answer(d["text"], ids, names):
            continue
        out.append(cid)
        if len(out) == k:
            break
    assert len(out) == k, f"{q['query_id']}: không đủ đoạn nhiễu"
    return out

BRIDGE   = {q["query_id"]: bridge_note(q) for q in QUERIES}
DISTRACT = {q["query_id"]: distractors(q, N_PASSAGES) for q in QUERIES}
print(f"✓ Đoạn nhiễu oracle: {len(DISTRACT)} câu; {sum(1 for v in BRIDGE.values() if v)} câu có ghi chú cầu nối (notebook 07)")

✓ Đoạn nhiễu oracle: 361 câu; 191 câu có ghi chú cầu nối (notebook 07)


## 4. Truy hồi trên kho trộn (E4)

**Kho trộn** (notebook 03): hệ thống nạp thêm bản mới mà không xoá bản cũ. Đoạn giống hệt nhau ở hai bản chỉ giữ một bản; đoạn khác nhau giữ cả hai. Phiên bản chỉ nằm ở `metadata`, không nằm trong văn bản. Ô đầu tiên **dựng lại** kho trộn từ hai kho của notebook 02 và kiểm tra trùng khớp từng đoạn.

**Bảy cấu hình** (mọi cấu hình RAG lấy **4 đoạn**, mỗi đoạn tối đa 250 từ; câu hỏi gửi cho mô hình luôn là câu hỏi gốc):

| Cấu hình | Truy hồi | Ngữ cảnh đưa cho mô hình |
|---|---|---|
| `closed` | không | không có ngữ cảnh (chỉ tri thức sẵn có) |
| `mixed` | BM25 trên kho trộn, truy vấn = câu hỏi | 4 đoạn theo thứ hạng, không nhãn |
| `mixed_label` | như `mixed` | cùng 4 đoạn, gắn nhãn tin cậy từ metadata: `vX (superseded)` / `vY (current release)` |
| `mixed_label_curfirst` | như `mixed` | như `mixed_label`, nhưng xếp đoạn bản hiện hành lên trước |
| `filter` | BM25 trên kho trộn, **lọc metadata** giữ đoạn có trong bản hiện hành | 4 đoạn, không nhãn |
| `filter_exp` | như `filter`, nhưng truy vấn được **mở rộng**: nếu câu hỏi nhắc mã (hoặc tên trong ngoặc kép) đã bị thu hồi, nối thêm mã và tên kế nhiệm | 4 đoạn, không nhãn |
| `filter_exp_note` | như `filter_exp` | 4 đoạn + **ghi chú dòng dõi** ở cuối: *"In MITRE ATT&CK vY, technique … was revoked and replaced by …"* |

Ghi chú dòng dõi được sinh từ bản đồ dòng dõi theo mã có trong câu hỏi, nên với câu có mã bị thu hồi nó **trùng từng ký tự** với ghi chú cầu nối của notebook 07 (ô kiểm tra xác nhận).

Bảng cuối mục là **chất lượng truy hồi** (chưa có LLM):

- `new_ev`: ngữ cảnh có đoạn bằng chứng của bản hiện hành;
- `old_only`: ngữ cảnh có ít nhất một đoạn chỉ có ở bản cũ;
- `old_ev`: ngữ cảnh có đoạn bằng chứng của bản cũ (khi nó khác bản mới);
- `note`: có ghi chú dòng dõi.

In [4]:
MIXED = {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_mixed"], encoding="utf-8"))}

# Dựng lại kho trộn theo đúng quy tắc của notebook 03 và so từng đoạn
_rebuilt = {}
for cid in sorted(set(CORP[V_OLD]) | set(CORP[V_NEW])):
    a, b = CORP[V_OLD].get(cid), CORP[V_NEW].get(cid)
    if a and b and a["text"] == b["text"] and a["title"] == b["title"]:
        _rebuilt[f"*|{cid}"] = (b["title"], b["text"], (V_OLD, V_NEW), cid)
    else:
        for v, c in ((V_OLD, a), (V_NEW, b)):
            if c:
                _rebuilt[f"{v}|{cid}"] = (c["title"], c["text"], (v,), cid)
assert set(_rebuilt) == set(MIXED), "Kho trộn khác kho dựng lại — gửi thầy"
for i, (ti, tx, vs, cid) in _rebuilt.items():
    m = MIXED[i]
    assert (m["title"], m["text"], tuple(m["metadata"]["versions"]), m["metadata"]["chunk_id"]) == (ti, tx, vs, cid), i
IS_CUR = {i: V_NEW in d["metadata"]["versions"] for i, d in MIXED.items()}
INDEX_MIX = BM25({i: d["title"] + "\n" + d["text"] for i, d in MIXED.items()})
print(f"✓ Kho trộn: {len(MIXED)} đoạn ({sum(len(d['metadata']['versions']) == 2 for d in MIXED.values())} chung, "
      f"{sum(d['metadata']['versions'] == [V_OLD] for d in MIXED.values())} chỉ bản cũ, "
      f"{sum(d['metadata']['versions'] == [V_NEW] for d in MIXED.values())} chỉ bản mới) — khớp dựng lại")

def mid(v, cid):                          # mã đoạn trong kho trộn
    return f"*|{cid}" if f"*|{cid}" in MIXED else f"{v}|{cid}"

ID_RE = re.compile(r"\bT\d{4}(?:\.\d{3})?\b")
def revoked_mentions(question):
    # Mã kỹ thuật bị thu hồi nhắc trong câu hỏi: theo mã, hoặc theo tên trong ngoặc kép (câu D2 hỏi theo tên)
    found = []
    for i in ID_RE.findall(question):
        if i in LINEAGE and i not in found:
            found.append(i)
    for n in re.findall(r'"([^"]+)"', question):
        i = NAME2OLD.get(n)
        if i and i not in found:
            found.append(i)
    return found

def expanded_query(q):
    add = [f"{s} {n}" for i in revoked_mentions(q["question"]) for s, n in LINEAGE[i]["succ"]]
    return q["question"] + (" " + " ".join(add) if add else "")

def lineage_note(q):
    found = revoked_mentions(q["question"])
    if not found:
        return None
    major = V_NEW.split(".")[0]
    return " ".join(f"In MITRE ATT&CK v{major}, technique {i} ({LINEAGE[i]['name_old']}) was revoked and replaced by "
                    + ", ".join(f"{s} ({n})" for s, n in LINEAGE[i]["succ"]) + "." for i in found)

def retrieve(query, pool, k=TOP_K):
    # Xếp hạng toàn kho trộn; pool="current": lọc metadata (giữ đoạn có trong bản hiện hành), rồi lấy k đoạn đầu
    ids = [i for i, _ in INDEX_MIX.search(query, len(MIXED)) if pool == "mixed" or IS_CUR[i]]
    assert len(ids) >= k
    return ids[:k]

RETR = {}
for q in QUERIES:
    qid = q["query_id"]
    RETR[("mixed", "question", qid)] = retrieve(q["question"], "mixed")
    RETR[("current", "question", qid)] = retrieve(q["question"], "current")
    RETR[("current", "expanded", qid)] = retrieve(expanded_query(q), "current")
NOTE = {q["query_id"]: lineage_note(q) for q in QUERIES}

# Kiểm tra: ghi chú dòng dõi trùng ghi chú cầu nối của notebook 07 ở mọi câu có ghi chú cầu nối
_mis = [k for k, v in BRIDGE.items() if v and NOTE[k] != v]
assert not _mis, f"Ghi chú dòng dõi khác ghi chú cầu nối: {_mis[:5]}"
_extra = [k for k in NOTE if NOTE[k] and not BRIDGE[k]]
print(f"✓ Ghi chú dòng dõi: {sum(1 for v in NOTE.values() if v)} câu; trùng ghi chú cầu nối ở "
      f"{sum(1 for v in BRIDGE.values() if v)} câu changed có mã bị thu hồi; {len(_extra)} câu khác cũng nhắc mã bị thu hồi"
      + (f" (ví dụ {_extra[:3]})" if _extra else ""))
assert all(IS_CUR[i] for (pool, _, _), ids in RETR.items() if pool == "current" for i in ids), "lọc metadata sai"
assert all((RETR[("current", "expanded", k)] == RETR[("current", "question", k)]) or revoked_mentions(QBYID[k]["question"])
           for k in QBYID), "truy vấn mở rộng đổi kết quả của câu không có mã bị thu hồi"

✓ Kho trộn: 4642 đoạn (3175 chung, 625 chỉ bản cũ, 842 chỉ bản mới) — khớp dựng lại
✓ Ghi chú dòng dõi: 191 câu; trùng ghi chú cầu nối ở 191 câu changed có mã bị thu hồi; 0 câu khác cũng nhắc mã bị thu hồi


In [5]:
def ctx_ids(q, cond):
    spec = CONDITIONS[cond]
    ids = list(RETR[(spec["pool"], spec["query"], q["query_id"])])
    if spec["order"] == "current_first":
        ids = [i for i in ids if IS_CUR[i]] + [i for i in ids if not IS_CUR[i]]
    return ids

DIAG = []
for q in QUERIES:
    qid = q["query_id"]
    eo, en = EV[qid]
    ev_new = {mid(V_NEW, c) for c in en}
    ev_old = {mid(V_OLD, c) for c in eo} - ev_new
    for cond in RAG_CONDS:
        ids = ctx_ids(q, cond)
        DIAG.append(dict(condition=cond, query_id=qid, template=q["template"], group=q["group"], revoked=REVOKED[qid],
                         new_ev=bool(ev_new & set(ids)), old_ev=bool(ev_old & set(ids)),
                         old_only=any(not IS_CUR[i] for i in ids), n_old_only=sum(not IS_CUR[i] for i in ids),
                         note=bool(CONDITIONS[cond]["note"] and NOTE[qid])))
DIAG = pd.DataFrame(DIAG)
DIAG.to_csv(OUT / "retrieval.csv", index=False)
RT = (DIAG[~DIAG.condition.isin(["mixed_label", "mixed_label_curfirst"])]    # cùng 4 đoạn với `mixed`
      .groupby(["group", "revoked", "condition"], sort=False)
      .agg(n=("query_id", "count"), new_ev=("new_ev", "mean"), old_only=("old_only", "mean"),
           old_ev=("old_ev", "mean"), note=("note", "mean")).round(2))
RT

n  new_ev  old_only  old_ev  note
group   revoked condition                                           
stable  False   mixed            160    0.91      0.98    0.67   0.0
                filter           160    0.98      0.00    0.00   0.0
                filter_exp       160    0.98      0.00    0.00   0.0
                filter_exp_note  160    0.98      0.00    0.00   0.0
changed True    mixed            191    0.51      1.00    0.98   0.0
                filter           191    0.79      0.00    0.00   0.0
                filter_exp       191    0.99      0.00    0.00   0.0
                filter_exp_note  191    0.99      0.00    0.00   1.0
        False   mixed             10    0.90      1.00    0.60   0.0
                filter            10    0.90      0.00    0.00   0.0
                filter_exp        10    0.90      0.00    0.00   0.0
                filter_exp_note   10    0.90      0.00    0.00   0.0

## 5. Prompt

Lời dặn hệ thống, yêu cầu theo kiểu đáp án và lược đồ JSON **giống hệt** notebook 04–08. Hai điểm khác:

- `closed`: chỉ có câu hỏi, không có ngữ cảnh (giống điều kiện `no_rag` của notebook 04).
- `cf_*` (E5): lời dặn hệ thống có thêm một câu cho phép báo mâu thuẫn, và lược đồ JSON có thêm trường bắt buộc `"conflict"` (true/false), **sau** trường `"answer"`. Ngữ cảnh giống hệt `old_new` / `new_old`.

Ô kiểm tra xác nhận:

- mỗi cấu hình RAG có đúng 4 đoạn (cộng ghi chú dòng dõi nếu có), nhãn đúng phiên bản trong metadata;
- prompt `old_new`, `new_old` trùng **từng byte** với notebook 07 (cả bộ prompt ghép lời dặn);
- prompt `cf_*` chỉ khác `old_new` / `new_old` ở lời dặn hệ thống và lược đồ.

In [6]:
SYSTEM = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
          "Answer the user's question accurately. Respond only with a JSON object of the form {\"answer\": [...]}. "
          "Put only the requested items in the list. Use an empty list if you do not know.")
SYSTEM_CF = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
             "Answer the user's question accurately. Respond only with a JSON object of the form "
             "{\"answer\": [...], \"conflict\": true or false}. "
             "Put only the requested items in the list. Use an empty list if you do not know. "
             "Set \"conflict\" to true if the context passages disagree with each other about the answer, otherwise false.")
INSTR = {
    "tactic_set":     'List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".',
    "technique_set":  "List ATT&CK technique IDs only (format T1234 or T1234.001).",
    "mitigation_set": "List ATT&CK mitigation IDs only (format M1234).",
}
PATTERN = {"technique_set": "^T[0-9]{4}(\\.[0-9]{3})?$", "mitigation_set": "^M[0-9]{4}$",
           "tactic_set": "^[A-Za-z0-9 ,.()&/'-]{1,60}$"}
TAGS = {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]",
        V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]"}   # = nhãn "full" của notebook 07

def schema_for(answer_type, conflict=False):
    props = {"answer": {"type": "array", "items": {"type": "string", "pattern": PATTERN[answer_type]}, "maxItems": 8}}
    if not conflict:
        return {"type": "object", "properties": props, "required": ["answer"], "additionalProperties": False}
    return {"type": "object", "properties": {**props, "conflict": {"type": "boolean"}},
            "required": ["answer", "conflict"], "additionalProperties": False}

def clip(text, n):
    words = list(re.finditer(r"\S+", text))
    return text if len(words) <= n else text[:words[n - 1].end()] + " …"

def passages(q, cond):
    # -> [(loại, mã đoạn hoặc phiên bản, nhãn, văn bản hiển thị)]
    spec, qid = CONDITIONS[cond], q["query_id"]
    if spec["kind"] == "closed":
        return []
    if spec["kind"] == "oracle":                       # giống notebook 07–08 (không nhãn)
        eo, en = EV[qid]
        out = []
        for kind in spec["order"]:
            if kind == "O":
                out += [("O", V_OLD, CORP[V_OLD][e]["text"]) for e in eo]
            elif kind == "N":
                out += [("N", V_NEW, CORP[V_NEW][e]["text"]) for e in en]
        out += [("D", V_NEW, CORP[V_NEW][d]["text"]) for d in DISTRACT[qid][:max(0, N_PASSAGES - len(out))]]
        return [(k, v, None, clip(t, MAX_CHUNK_WORDS)) for k, v, t in out]
    res = []
    for i in ctx_ids(q, cond):
        label = TAGS[V_NEW if IS_CUR[i] else V_OLD] if spec["label"] else None
        res.append(("R", i, label, (label + "\n" if label else "") + clip(MIXED[i]["text"], MAX_CHUNK_WORDS)))
    if spec["note"] and NOTE[qid]:
        res.append(("B", None, None, NOTE[qid]))
    return res

def build_messages(q, cond):
    spec = CONDITIONS[cond]
    system = SYSTEM_CF if spec.get("conflict") else SYSTEM
    if spec["kind"] == "closed":
        user = f"Question: {q['question']}\n{INSTR[q['answer_type']]}"
    else:
        ctx = "\n\n".join(f"[{i + 1}] {t}" for i, (_, _, _, t) in enumerate(passages(q, cond)))
        user = ("Use the following context passages from a CTI knowledge base if they are relevant.\n\n"
                f"{ctx}\n\nQuestion: {q['question']}\n{INSTR[q['answer_type']]}")
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def merge_system(msgs):
    return [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]

def prompt_row(q, cond):
    spec = CONDITIONS[cond]
    row = dict(query_id=q["query_id"], condition=cond, answer_type=q["answer_type"], messages=build_messages(q, cond),
               schema=schema_for(q["answer_type"], spec.get("conflict", False)))
    if spec["kind"] == "oracle" and not spec["conflict"]:
        row["layout"] = [k for k, *_ in passages(q, cond)]        # đúng các trường của notebook 07
    elif spec["kind"] != "closed":
        row["layout"] = [k for k, *_ in passages(q, cond)]
        if spec["kind"] == "rag":
            row["retrieved"] = ctx_ids(q, cond)
    return row

for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as f1, \
         open(PROMPTS_MS / f"{cond}.jsonl", "w", encoding="utf-8") as f2:
        for q in QUERIES:
            row = prompt_row(q, cond)
            f1.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
            f2.write(json.dumps({**row, "messages": merge_system(row["messages"])}, ensure_ascii=False, sort_keys=True) + "\n")
MANIFEST["prompts"] = {c: sha256(PROMPTS / f"{c}.jsonl") for c in CONDITIONS}
MANIFEST["prompts_mergedsys"] = {c: sha256(PROMPTS_MS / f"{c}.jsonl") for c in CONDITIONS}
_w = {c: max(len(build_messages(q, c)[1]["content"].split()) for q in QUERIES) for c in CONDITIONS}
print("Số từ lớn nhất của lượt người dùng theo cấu hình:", _w)

Số từ lớn nhất của lượt người dùng theo cấu hình: {'closed': 52, 'mixed': 1071, 'mixed_label': 1097, 'mixed_label_curfirst': 1097, 'filter': 1071, 'filter_exp': 1071, 'filter_exp_note': 1071, 'old_new': 1064, 'new_old': 1064, 'cf_old_new': 1064, 'cf_new_old': 1064}


In [7]:
errs = []
for q in QUERIES:
    qid = q["query_id"]
    for cond, spec in CONDITIONS.items():
        ps, msgs = passages(q, cond), build_messages(q, cond)
        text = msgs[1]["content"]
        if spec["kind"] == "rag":
            n_r = sum(k == "R" for k, *_ in ps)
            n_b = sum(k == "B" for k, *_ in ps)
            if n_r != TOP_K or n_b != (1 if spec["note"] and NOTE[qid] else 0) or (n_b and ps[-1][0] != "B"):
                errs.append((qid, cond, "số đoạn"))
            for k, i, label, _ in ps:
                if k == "R" and label != (TAGS[V_NEW if IS_CUR[i] else V_OLD] if spec["label"] else None):
                    errs.append((qid, cond, "nhãn"))
            if any(t in text for t in TAGS.values()) != spec["label"]:
                errs.append((qid, cond, "nhãn trong prompt"))
            if spec["order"] == "current_first":
                cur = [IS_CUR[i] for k, i, *_ in ps if k == "R"]
                if cur != sorted(cur, reverse=True) or sorted(i for k, i, *_ in ps if k == "R") != \
                        sorted(RETR[("mixed", "question", qid)]):
                    errs.append((qid, cond, "xếp bản hiện hành lên trước"))
        elif spec["kind"] == "closed":
            if "context passages" in text or "[1]" in text:
                errs.append((qid, cond, "closed có ngữ cảnh"))
        if f"Question: {q['question']}\n" not in text:
            errs.append((qid, cond, "câu hỏi gốc"))
    for a, b in (("cf_old_new", "old_new"), ("cf_new_old", "new_old")):   # cf_* chỉ khác lời dặn và lược đồ
        ra, rb = prompt_row(q, a), prompt_row(q, b)
        if ra["messages"][1] != rb["messages"][1] or ra["messages"][0]["content"] != SYSTEM_CF \
                or rb["messages"][0]["content"] != SYSTEM or ra["layout"] != rb["layout"]:
            errs.append((qid, a, "cf khác ngữ cảnh"))
assert not errs, errs[:10]
for cond in CONDITIONS:                                   # bộ prompt ghép lời dặn: cùng nội dung
    for r1, r2 in zip(map(json.loads, open(PROMPTS / f"{cond}.jsonl", encoding="utf-8")),
                      map(json.loads, open(PROMPTS_MS / f"{cond}.jsonl", encoding="utf-8"))):
        assert r1["query_id"] == r2["query_id"] and len(r2["messages"]) == 1 and r2["messages"][0]["role"] == "user"
        assert r2["messages"][0]["content"] == r1["messages"][0]["content"] + "\n\n" + r1["messages"][1]["content"]
print(f"✓ {len(QUERIES) * len(CONDITIONS)} prompt đúng cấu trúc (mỗi bộ)")

# old_new, new_old: trùng từng byte với notebook 07 (cả hai bộ prompt)
for key in ("prompts", "prompts_mergedsys"):
    for c in ("old_new", "new_old"):
        assert M07[key][c] == MANIFEST[key][c], f"{key}/{c}.jsonl khác notebook 07 — gửi thầy"
print("✓ Prompt old_new, new_old trùng từng byte với notebook 07 (cả bộ ghép lời dặn)")

✓ 3971 prompt đúng cấu trúc (mỗi bộ)
✓ Prompt old_new, new_old trùng từng byte với notebook 07 (cả bộ ghép lời dặn)


In [8]:
# Bộ chạy thử: 1 câu cho mỗi (mẫu, nhóm, có mã bị thu hồi) -> có cả câu có ghi chú dòng dõi
smoke_ids = set()
for key in sorted({(q["template"], q["group"], REVOKED[q["query_id"]]) for q in QUERIES}):
    same = [q["query_id"] for q in QUERIES if (q["template"], q["group"], REVOKED[q["query_id"]]) == key]
    smoke_ids |= set(same[:SMOKE_PER_GROUP])
for src, dst in ((PROMPTS, SMOKE_P), (PROMPTS_MS, SMOKE_P_MS)):
    for cond in CONDITIONS:
        with open(src / f"{cond}.jsonl", encoding="utf-8") as fi, open(dst / f"{cond}.jsonl", "w", encoding="utf-8") as fo:
            for line in fi:
                if json.loads(line)["query_id"] in smoke_ids:
                    fo.write(line)
print(f"Bộ chạy thử: {len(smoke_ids)} câu × {len(CONDITIONS)} cấu hình")
_ex = next(q for q in QUERIES if q["group"] == "changed" and NOTE[q["query_id"]])
print("\nVí dụ prompt filter_exp_note (lượt người dùng):\n" + build_messages(_ex, "filter_exp_note")[1]["content"][-900:])

Bộ chạy thử: 5 câu × 11 cấu hình

Ví dụ prompt filter_exp_note (lượt người dùng):
ollection: FIN6 has used a script to iterate through a list of compromised PoS systems, copy data to a log file, and remove the original data files.
Uses T1547.001 Registry Run Keys / Startup Folder: FIN6 has used Registry Run keys to establish persistence for its downloader tools known as HARDTACK and SHIPBREAD.
Uses T1553.002 Code Signing: FIN6 has used Comodo code-signing certificates.
Uses T1560 Archive Collected Data: Following data collection, FIN6 has …

[5] In MITRE ATT&CK v7, technique T1002 (Data Compressed) was revoked and replaced by T1560 (Archive Collected Data).

Question: According to the current MITRE ATT&CK Enterprise release (v7.2), which tactic(s) does the technique Data Compressed (T1002) belong to? If that technique ID has been replaced, answer for its replacement(s).
List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".


## 6. Chạy suy luận

Như notebook 07–08: mỗi mô hình chạy trong một tiến trình riêng.

- **Chạy thử** trước cho mọi mô hình (vài phút mỗi mô hình), để phát hiện sớm lỗi tải mô hình hay lỗi định dạng.
- **Theo dõi tiến độ:** lệnh `tail -f …` được in ra để em xem trong terminal. Ô sẽ im lặng trong lúc chạy.
- **Mô hình đã chạy xong** (cùng prompt, cùng cấu hình) sẽ được bỏ qua nếu chạy lại ô. Nếu máy bị ngắt giữa chừng, em chạy lại từ đầu notebook; phần đã xong không chạy lại.
- **Nếu một mô hình chạy lỗi**, notebook tự thử các cấu hình dự phòng (mục 1). Log ở cấu hình k ≥ 1 có đuôi `_r<k>`.

Ô đầu tiên ghi module suy luận bản 2 (không đổi so với notebook 07–08) và kiểm tra mã băm. Ô thứ hai kiểm tra mô hình đã có trong `/data/hf` (nếu thiếu thì tải).

In [9]:
INFER_SRC = r'''# Chạy suy luận cho MỘT mô hình (bản 2, notebook 07). Gọi từ notebook qua subprocess.
# Khác bản 1 (notebook 04): thêm --engine-kwargs (tham số máy vLLM cho cấu hình môi trường dự phòng),
# ghi biến môi trường BLAS vào khoá chạy và meta, in bộ nhớ GPU trước khi nạp mô hình.
import argparse, hashlib, json, os, sys, time, platform, subprocess
from pathlib import Path

os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


def done_ok(path, meta_path, ids, run_key):
    # Kết quả cũ chỉ được dùng lại khi: đủ đúng các query_id theo đúng thứ tự VÀ cùng "khoá chạy"
    # (cùng file prompt, cùng mô hình/phiên bản, cùng tham số sinh). Đổi bất cứ thứ gì -> chạy lại.
    if not path.exists() or not meta_path.exists():
        return False
    try:
        got = [r["query_id"] for r in read_jsonl(path)]
        key = json.loads(meta_path.read_text()).get("run_key")
    except (ValueError, KeyError):
        return False
    return got == ids and key == run_key


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--revision", required=True)
    ap.add_argument("--tag", required=True)
    ap.add_argument("--prompts-dir", required=True)
    ap.add_argument("--conditions", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--max-model-len", type=int, default=6144)
    ap.add_argument("--max-tokens", type=int, default=96)
    ap.add_argument("--gpu-mem", type=float, default=0.85)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--chat-template-kwargs", default="{}")
    ap.add_argument("--engine-kwargs", default="{}")
    a = ap.parse_args()

    conds = [c for c in a.conditions.split(",") if c]
    out_dir = Path(a.out_dir) / a.tag
    out_dir.mkdir(parents=True, exist_ok=True)
    ctk = json.loads(a.chat_template_kwargs)
    ek = json.loads(a.engine_kwargs)
    assert isinstance(ek, dict) and set(ek) <= {"enforce_eager", "max_num_seqs", "max_num_batched_tokens"}, ek
    blas_env = os.environ.get("TORCH_BLAS_PREFER_CUBLASLT", "")

    from importlib.metadata import version as pkg_version
    script_sha = sha256_file(Path(__file__))
    vllm_ver = pkg_version("vllm")
    jobs, keys = {}, {}
    for c in conds:
        pfile = Path(a.prompts_dir) / f"{c}.jsonl"
        rows = read_jsonl(pfile)
        if a.limit:
            rows = rows[:a.limit]
        ids = [r["query_id"] for r in rows]
        keys[c] = hashlib.sha256(json.dumps(dict(prompts=sha256_file(pfile), limit=a.limit, repo=a.repo,
                                                 revision=a.revision, max_model_len=a.max_model_len,
                                                 max_tokens=a.max_tokens, seed=a.seed, ctk=ctk, gpu_mem=a.gpu_mem,
                                                 script=script_sha, vllm=vllm_ver, engine=ek, blas_env=blas_env),
                                            sort_keys=True).encode()).hexdigest()
        if done_ok(out_dir / f"{c}.jsonl", out_dir / f"meta_{c}.json", ids, keys[c]):
            print(f"[{a.tag}] {c}: đã có đủ {len(ids)} kết quả cùng cấu hình — bỏ qua", flush=True)
            continue
        jobs[c] = rows
    if not jobs:
        print(f"[{a.tag}] không còn việc gì để chạy", flush=True)
        return

    from huggingface_hub import snapshot_download
    path = snapshot_download(a.repo, revision=a.revision)
    print(f"[{a.tag}] mô hình tại {path}", flush=True)

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    budget = a.max_model_len - a.max_tokens
    for c, rows in jobs.items():
        lens = []
        for r in rows:
            text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True, **ctk)
            lens.append(len(tok(text, add_special_tokens=False)["input_ids"]))
        too_long = [(r["query_id"], n) for r, n in zip(rows, lens) if n > budget]
        print(f"[{a.tag}] {c}: {len(rows)} prompt, dài nhất {max(lens)} token (giới hạn {budget})", flush=True)
        if too_long:
            sys.exit(f"LỖI: {len(too_long)} prompt vượt {budget} token, ví dụ {too_long[:3]} — tăng MAX_MODEL_LEN "
                     "hoặc giảm MAX_CHUNK_WORDS trong notebook")

    try:   # nvidia-smi không khởi tạo CUDA trong tiến trình này (vLLM cần điều đó)
        smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        smi = f"không đọc được ({type(e).__name__})"
    print(f"[{a.tag}] GPU trước khi nạp: {smi}; TORCH_BLAS_PREFER_CUBLASLT={blas_env or '-'}; engine={ek}", flush=True)

    import torch
    import vllm
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams

    try:   # đọc ở tiến trình này; EngineCore thừa hưởng cùng biến môi trường
        blas_backend = str(torch.backends.cuda.preferred_blas_library())
    except Exception as e:
        blas_backend = f"? ({type(e).__name__})"
    # disable_any_whitespace: JSON dạng gọn, mô hình không thể sinh khoảng trắng vô hạn rồi bị cắt ở max_tokens
    llm = LLM(model=path, tokenizer=path, max_model_len=a.max_model_len,
              gpu_memory_utilization=a.gpu_mem, seed=a.seed,
              structured_outputs_config={"backend": "xgrammar", "disable_any_whitespace": True}, **ek)
    meta_base = dict(model_tag=a.tag, repo=a.repo, revision=a.revision, snapshot=Path(path).name,
                     vllm=vllm.__version__, torch=torch.__version__, cuda=torch.version.cuda,
                     gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
                     max_model_len=a.max_model_len, max_tokens=a.max_tokens, gpu_mem=a.gpu_mem, seed=a.seed,
                     chat_template_kwargs=ctk, flashinfer_sampler=os.environ.get("VLLM_USE_FLASHINFER_SAMPLER"),
                     script_sha256=script_sha, structured_outputs="xgrammar, disable_any_whitespace",
                     engine_kwargs=ek, blas_env=blas_env, blas_backend_parent=blas_backend)

    for c, rows in jobs.items():
        sps = [SamplingParams(temperature=0.0, max_tokens=a.max_tokens, seed=a.seed,
                              structured_outputs=StructuredOutputsParams(json=r["schema"])) for r in rows]
        t0 = time.time()
        outs = llm.chat([r["messages"] for r in rows], sps, use_tqdm=True, chat_template_kwargs=ctk or None)
        dt = time.time() - t0
        assert len(outs) == len(rows)
        tmp = out_dir / f".{c}.jsonl.tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for r, o in zip(rows, outs):
                f.write(json.dumps(dict(query_id=r["query_id"], condition=c, model_tag=a.tag,
                                        text=o.outputs[0].text, finish_reason=o.outputs[0].finish_reason,
                                        n_prompt_tokens=len(o.prompt_token_ids or []),
                                        n_output_tokens=len(o.outputs[0].token_ids)), ensure_ascii=False) + "\n")
        os.replace(tmp, out_dir / f"{c}.jsonl")
        (out_dir / f"meta_{c}.json").write_text(json.dumps({**meta_base, "condition": c, "n": len(rows),
                                                            "run_key": keys[c], "seconds": round(dt, 1)}, indent=2))
        print(f"[{a.tag}] {c}: xong {len(rows)} truy vấn trong {dt / 60:.1f} phút", flush=True)


if __name__ == "__main__":
    main()
'''

INFER = ROOT / "ctidrift_infer_v2.py"
if not INFER.exists() or sha256(INFER) != "39b80fe37bb06d4ddbe082a6c372258afe10967a0a599539cd2449d822ea4f74":
    INFER.write_text(INFER_SRC, encoding="utf-8")
assert sha256(INFER) == "39b80fe37bb06d4ddbe082a6c372258afe10967a0a599539cd2449d822ea4f74", "ctidrift_infer_v2.py bị sửa — gửi thầy"
MANIFEST["inputs"]["infer_module"] = {"path": INFER.name, "sha256": sha256(INFER)}
print(f"✓ {INFER.name} ({sha256(INFER)[:12]}…)")

✓ ctidrift_infer_v2.py (39b80fe37bb0…)


In [10]:
SNAP = {}
for m in MODELS:
    _r = subprocess.run([sys.executable, "-c", f"from huggingface_hub import snapshot_download; "
                         f"print(snapshot_download({m['repo']!r}, revision={m['revision']!r}))"],
                        env=ENV, capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("Tải mô hình lỗi (mạng hoặc ổ đĩa) — kiểm tra rồi chạy lại ô này. Thông báo:\n" + _r.stderr[-2000:])
    SNAP[m["tag"]] = _r.stdout.strip().splitlines()[-1]
    print(f"✓ {m['tag']:16s} {SNAP[m['tag']]}")

✓ qwen3-8b         /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
✓ llama3.1-8b      /data/hf/hub/models--hugging-quants--Meta-Llama-3.1-8B-Instruct-AWQ-INT4/snapshots/db1f81ad4b8c7e39777509fac66c652eb0a52f91
✓ gemma2-9b        /data/hf/hub/models--hugging-quants--gemma-2-9b-it-AWQ-INT4/snapshots/6e62725da8e92309167814dad7aacc0ed8cb2484
✓ phi4-mini        /data/hf/hub/models--microsoft--Phi-4-mini-instruct/snapshots/cfbefacb99257ffa30c83adab238a50856ac3083
✓ mistral7b-v0.3   /data/hf/hub/models--solidrust--Mistral-7B-Instruct-v0.3-AWQ/snapshots/95b1295ddd1a8673117cdc7bd2a4da2a457bb3f7
✓ qwen3-14b        /data/hf/hub/models--Qwen--Qwen3-14B-AWQ/snapshots/31c69efc29464b6bb0aee1398b5a7b50a99340c3


In [11]:
class InferError(RuntimeError):
    pass

def ladder(model):
    return LADDERS[model["ladder"]]

def effective(model, k):
    R = ladder(model)[k]
    return dict(gpu_mem=R["gpu_mem"], max_model_len=min(model.get("max_model_len", R["max_model_len"]), R["max_model_len"]))

def _group_alive(pgid):
    for d in os.listdir("/proc"):
        if not d.isdigit():
            continue
        try:
            with open(f"/proc/{d}/stat") as fh:
                st = fh.read()
        except OSError:
            continue
        rest = st[st.rindex(")") + 2:].split()
        if int(rest[2]) == pgid and rest[0] not in ("Z", "X"):
            return True
    return False

def run_infer(model, out_dir, smoke=False, rung=0):
    R = ladder(model)[rung]
    pdir = (SMOKE_P_MS if smoke else PROMPTS_MS) if model["merge_system"] else (SMOKE_P if smoke else PROMPTS)
    log = LOGS / f"{model['tag']}{'_smoke' if smoke else ''}{f'_r{rung}' if rung else ''}.log"
    eff = effective(model, rung)
    cmd = [sys.executable, INFER.name, "--repo", model["repo"], "--revision", model["revision"],
           "--tag", model["tag"], "--prompts-dir", str(pdir), "--conditions", ",".join(CONDITIONS),
           "--out-dir", str(out_dir), "--max-model-len", str(eff["max_model_len"]),
           "--max-tokens", str(MAX_TOKENS),
           "--gpu-mem", str(eff["gpu_mem"]), "--seed", str(SEED),
           "--chat-template-kwargs", json.dumps(model["chat_template_kwargs"]),
           "--engine-kwargs", json.dumps(R["engine"], sort_keys=True)]
    env = {**ENV, **R["env"]}
    print(f"▶ {model['tag']} [cấu hình {rung}: {R['name']}]: đang chạy… "
          f"theo dõi tiến độ trong terminal bằng lệnh:\n    tail -f {log}", flush=True)
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=ROOT, start_new_session=True)
        def _kill(sig):
            try:
                os.killpg(p.pid, sig)
            except ProcessLookupError:
                pass
        try:
            p.wait()
        except BaseException:
            _kill(signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except BaseException:
                pass
            _kill(signal.SIGKILL)
            print("⏹ Đã dừng và giải phóng GPU. Kiểm tra lại bằng lệnh nvidia-smi nếu cần.")
            raise
    _kill(signal.SIGKILL)
    t1 = time.time()
    while _group_alive(p.pid) and time.time() - t1 < 120:
        time.sleep(1)
    if _group_alive(p.pid):
        print(f"⚠ Tiến trình của {model['tag']} chưa thoát hẳn sau 120 giây — kiểm tra bằng nvidia-smi")
    time.sleep(3)
    text = log.read_text(encoding="utf-8", errors="replace")
    tail = text.splitlines()
    print("\n".join(l for l in tail if l.startswith(f"[{model['tag']}]") or "LỖI" in l or "Error" in l)[-3000:])
    print(f"◀ {model['tag']}: {'XONG' if p.returncode == 0 else 'LỖI'} sau {(time.time() - t0) / 60:.1f} phút")
    if p.returncode != 0:
        print("\n".join(tail[-40:]))
        if "prompt vượt" in text:      # lỗi cấu hình của notebook, không phải lỗi môi trường: dừng hẳn
            raise AssertionError(f"Prompt dài quá giới hạn — gửi thầy file {log}")
        raise InferError(f"Suy luận lỗi — gửi thầy file {log}")

MBYTAG = {m["tag"]: m for m in MODELS}

def try_ladder(model, out_dir, start, smoke):
    for k in range(start, len(ladder(model))):
        rec = dict(phase="chạy thử" if smoke else "chạy chính", rung=k)
        try:
            run_infer(model, out_dir, smoke=smoke, rung=k)
        except InferError as e:
            ATTEMPTS.setdefault(model["tag"], []).append({**rec, "ok": False, "error": str(e)})
            if k + 1 < len(ladder(model)):
                if smoke:                            # chạy thử hỏng: lần chạy lại không bắt đầu thấp hơn cấu hình kế tiếp
                    START[model["tag"]] = max(START.get(model["tag"], 0), k + 1)
                elif model["tag"] in RUNG:           # chạy chính hỏng: ghi ngay, để lần chạy lại không quay xuống
                    RUNG[model["tag"]] = k + 1
            save_env_rung()
            if k + 1 < len(ladder(model)):
                print(f"↻ {model['tag']}: thử cấu hình dự phòng {k + 1} ({ladder(model)[k + 1]['name']})", flush=True)
            continue
        ATTEMPTS.setdefault(model["tag"], []).append({**rec, "ok": True})
        RUNG[model["tag"]] = k
        save_env_rung()
        return k
    return None

def save_env_rung():
    cur = {**START, **RUNG}
    MANIFEST["env_rung"] = {t: dict(rung=k, ladder=MBYTAG[t]["ladder"], **ladder(MBYTAG[t])[k],
                                    effective=effective(MBYTAG[t], k)) for t, k in cur.items()}
    MANIFEST["env_attempts"] = ATTEMPTS
    tmp = OUT / ".env_rung.json.tmp"
    tmp.write_text(json.dumps(dict(rung=MANIFEST["env_rung"], start=START, attempts=ATTEMPTS), indent=2, ensure_ascii=False))
    os.replace(tmp, OUT / "env_rung.json")

def read_rungs(p):
    if not p.exists():
        return {}, {}
    d = json.loads(p.read_text())
    return {t: v["rung"] for t, v in d.get("rung", {}).items()}, d.get("attempts", {})

# Cấu hình bắt đầu (chỉ đi lên): lần chạy trước của notebook này; cấu hình đã dùng ở notebook 07/08 cùng nhánh;
# ở nhánh B: cấu hình đã dùng ở nhánh A của notebook này.
START, ATTEMPTS = read_rungs(OUT / "env_rung.json")
START = {t: k for t, k in START.items() if t in MBYTAG}
_prior = {**read_rungs(CONF07 / "env_rung.json")[0], **read_rungs(EXT08 / "env_rung.json")[0]}
if CFG["branch"] == "B":
    _a = ROOT / "results" / "e2e-18.1-19.2" / "env_rung.json"
    if _a.exists():
        for t, k in read_rungs(_a)[0].items():
            _prior[t] = max(_prior.get(t, 0), k)
    else:
        print("⚠ Chưa có kết quả nhánh A của notebook này — theo kế hoạch, nên chạy nhánh A trước")
for t, k in _prior.items():
    if t in MBYTAG:
        START[t] = max(START.get(t, 0), k)
print("Cấu hình bắt đầu:", {m["tag"]: START.get(m["tag"], 0) for m in MODELS})

def conflict_flag(text):
    # Giá trị "conflict" của đối tượng JSON cuối cùng có khoá "answer"; None nếu không có
    for obj in reversed(_json_objects(text or "")):
        low = {str(k).lower(): v for k, v in obj.items()}
        if "answer" in low:
            return low["conflict"] if isinstance(low.get("conflict"), bool) else None
    return None

def leftover_infer():
    # Tiến trình suy luận còn chạy từ lần trước (ví dụ sau khi khởi động lại kernel): chiếm GPU và có thể ghi đè kết quả
    out = []
    for d in os.listdir("/proc"):
        if d.isdigit() and int(d) != os.getpid():
            try:
                cmd = open(f"/proc/{d}/cmdline", "rb").read().replace(b"\0", b" ").decode(errors="replace")
            except OSError:
                continue
            if "ctidrift_infer_v2.py" in cmd and "--out-dir" in cmd:
                out.append((int(d), cmd[:120]))
    return out
# Đo độ dài MỌI prompt bằng tokenizer của từng mô hình (không cần GPU, khoảng 1 phút mỗi mô hình), cùng cách tính với
# module suy luận, để phát hiện prompt quá dài TRƯỚC khi chạy, thay vì sau vài giờ
LEN_SRC = r'''
import json, sys
from pathlib import Path
from transformers import AutoTokenizer
path, pdir, ctk, conds = sys.argv[1], sys.argv[2], json.loads(sys.argv[3]), sys.argv[4].split(",")
tok = AutoTokenizer.from_pretrained(path)
mx = {}
for c in conds:
    n = 0
    for l in open(Path(pdir) / f"{c}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        t = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True, **ctk)
        n = max(n, len(tok(t, add_special_tokens=False)["input_ids"]))
    mx[c] = n
print(json.dumps(mx))
'''
PLEN = {}
for m in MODELS:
    pdir = PROMPTS_MS if m["merge_system"] else PROMPTS
    _r = subprocess.run([sys.executable, "-c", LEN_SRC, SNAP[m["tag"]], str(pdir), json.dumps(m["chat_template_kwargs"]),
                         ",".join(CONDITIONS)], env=ENV, capture_output=True, text=True)
    assert _r.returncode == 0, f"Không đo được độ dài prompt của {m['tag']}:\n" + _r.stderr[-2000:]
    PLEN[m["tag"]] = json.loads(_r.stdout.strip().splitlines()[-1])
    k0 = START.get(m["tag"], 0)
    budgets = [effective(m, k)["max_model_len"] - MAX_TOKENS for k in range(k0, len(ladder(m)))]
    longest = max(PLEN[m["tag"]].values())
    print(f"  {m['tag']:16s} prompt dài nhất {longest} token; giới hạn ở cấu hình {k0}: {budgets[0]}, "
          f"thấp nhất trên thang: {min(budgets)}")
    assert longest <= budgets[0], f"Prompt của {m['tag']} dài quá giới hạn ({longest} > {budgets[0]}) — gửi thầy"
    if longest > min(budgets):
        print(f"  ⚠ {m['tag']}: nếu phải lên cấu hình dự phòng cao hơn, prompt sẽ dài quá giới hạn — gửi thầy nếu thấy dòng này")
MANIFEST["prompt_tokens_max"] = PLEN
print("✓ Mọi prompt nằm trong giới hạn token ở cấu hình bắt đầu")

_left = leftover_infer()
assert not _left, ("Còn tiến trình suy luận của lần chạy trước (đang chiếm GPU): " + str(_left) +
                   " — tắt chúng bằng lệnh kill <pid> (kiểm tra bằng nvidia-smi), rồi chạy lại ô này")

FAILED, RUNG = {}, {}
SMOKE = OUT / "raw_smoke"
for m in MODELS:
    k0 = START.get(m["tag"], 0)
    k = try_ladder(m, SMOKE, k0, smoke=True)
    if k is None:
        FAILED[m["tag"]] = f"chạy thử lỗi ở mọi cấu hình từ {k0} tới {len(ladder(m)) - 1}"
        continue
    rs = [(cond, r) for cond in CONDITIONS for r in map(json.loads, open(SMOKE / m["tag"] / f"{cond}.jsonl", encoding="utf-8"))]
    fmt = [score_drift(QBYID[r["query_id"]], r["text"], TACTIC_ALIAS)["format"] == "json" for _, r in rs]
    cf = [conflict_flag(r["text"]) is not None for c, r in rs if CONDITIONS[c].get("conflict")]
    if np.mean(fmt) < JSON_MIN:
        FAILED[m["tag"]] = f"chạy thử: tỉ lệ JSON {np.mean(fmt):.2f} < {JSON_MIN}"
    elif np.mean(cf) < 1:
        print(f"⚠ {m['tag']}: {len(cf) - sum(cf)}/{len(cf)} câu trả lời cf_* thiếu trường conflict (chỉ ảnh hưởng E5; vẫn chạy tiếp)")
save_env_rung()
for t, k in RUNG.items():
    print(f"  {t:16s} cấu hình {k}: {ladder(MBYTAG[t])[k]['name']}")
if FAILED:
    print("⚠⚠ CÓ MÔ HÌNH LỖI KỸ THUẬT — gửi thầy file log trước khi chạy tiếp:", FAILED)
else:
    print("✓ Chạy thử: mọi mô hình chạy được và trả lời đúng định dạng JSON")

Cấu hình bắt đầu: {'qwen3-8b': 0, 'llama3.1-8b': 0, 'gemma2-9b': 0, 'phi4-mini': 0, 'mistral7b-v0.3': 0, 'qwen3-14b': 2}
  qwen3-8b         prompt dài nhất 1963 token; giới hạn ở cấu hình 0: 5984, thấp nhất trên thang: 3936
  llama3.1-8b      prompt dài nhất 1941 token; giới hạn ở cấu hình 0: 5984, thấp nhất trên thang: 3936
  gemma2-9b        prompt dài nhất 2024 token; giới hạn ở cấu hình 0: 5984, thấp nhất trên thang: 3936
  phi4-mini        prompt dài nhất 1908 token; giới hạn ở cấu hình 0: 3936, thấp nhất trên thang: 3936
  mistral7b-v0.3   prompt dài nhất 2270 token; giới hạn ở cấu hình 0: 5984, thấp nhất trên thang: 3936
  qwen3-14b        prompt dài nhất 1963 token; giới hạn ở cấu hình 2: 2400, thấp nhất trên thang: 2400
✓ Mọi prompt nằm trong giới hạn token ở cấu hình bắt đầu
▶ qwen3-8b [cấu hình 0: mặc định]: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/e2e-6.3-7.2/logs/qwen3-8b_smoke.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Q

In [12]:
for m in MODELS:
    t = m["tag"]
    if t in FAILED:
        print(f"⏭ Bỏ qua {t} (lỗi kỹ thuật: {FAILED[t]})")
        continue
    assert t in RUNG, f"Chưa có kết quả chạy thử của {t} — chạy lại ô chạy thử ở trên trước"
    k = try_ladder(m, RAW, RUNG[t], smoke=False)
    if k is None:
        FAILED[t] = f"chạy chính lỗi ở mọi cấu hình từ {RUNG[t]} tới {len(ladder(m)) - 1}"
    else:
        RUNG[t] = k
save_env_rung()
(OUT / "tech_fail.json").write_text(json.dumps(FAILED, indent=2, ensure_ascii=False))
print("Cấu hình dùng cho chạy chính:", {t: f"{k}: {ladder(MBYTAG[t])[k]['name']}" for t, k in RUNG.items() if t not in FAILED})
print("Mô hình lỗi kỹ thuật:", FAILED or "không có")

▶ qwen3-8b [cấu hình 0: mặc định]: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/e2e-6.3-7.2/logs/qwen3-8b.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] closed: 361 prompt, dài nhất 167 token (giới hạn 5984)
[qwen3-8b] mixed: 361 prompt, dài nhất 1881 token (giới hạn 5984)
[qwen3-8b] mixed_label: 361 prompt, dài nhất 1951 token (giới hạn 5984)
[qwen3-8b] mixed_label_curfirst: 361 prompt, dài nhất 1951 token (giới hạn 5984)
[qwen3-8b] filter: 361 prompt, dài nhất 1963 token (giới hạn 5984)
[qwen3-8b] filter_exp: 361 prompt, dài nhất 1963 token (giới hạn 5984)
[qwen3-8b] filter_exp_note: 361 prompt, dài nhất 1963 token (giới hạn 5984)
[qwen3-8b] old_new: 361 prompt, dài nhất 1801 token (giới hạn 5984)
[qwen3-8b] new_old: 361 prompt, dài nhất 1801 token (giới hạn 5984)
[qwen3-8b] cf_old_new: 361 prompt, dài nhất 1831 token (giới hạn 5984)
[qwen3-8b] cf_new_old: 361 

## 7. Chấm điểm

Bộ chấm v1.3 cho mọi câu trả lời (`drift_ext` là chỉ số chính, `stale13` cho phân tích độ nhạy). Thêm các cột:

- `stale_union`: câu lỗi thời (có phần tử chỉ thuộc đáp án cũ) **và** có ít nhất một phần tử chỉ thuộc đáp án mới, tức gộp hai bản; `stale_old`: câu lỗi thời không có phần tử nào chỉ thuộc đáp án mới (với câu mà đáp án mới nằm trọn trong đáp án cũ, lỗi thời luôn thuộc loại này);
- `no_answer`: danh sách rỗng;
- `conflict`: giá trị trường `"conflict"` (chỉ ở `cf_*`).

In [13]:
def pack():
    MANIFEST["n_queries"] = len(QUERIES)
    MANIFEST["runs"] = {}
    for m in MODELS:
        for cond in CONDITIONS:
            mp = RAW / m["tag"] / f"meta_{cond}.json"
            if mp.exists():
                MANIFEST["runs"][f"{m['tag']}/{cond}"] = json.loads(mp.read_text())
    MANIFEST["outputs"] = {str(p.relative_to(OUT)): sha256(p) for p in sorted(OUT.rglob("*"))
                           if p.is_file() and p.suffix in (".csv", ".jsonl") and "raw_smoke" not in p.parts
                           and not p.parts[len(OUT.parts)].startswith("prompts")}
    (OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
    ZIP = OUT / "gui_thay.zip"
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob("*")):
            top = p.parts[len(OUT.parts)]
            if p.is_file() and p != ZIP and not top.startswith("prompts_smoke") and not top.startswith("prompts_mergedsys"):
                z.write(p, p.relative_to(OUT))
    print(f"Đã tạo {ZIP.relative_to(ROOT)} ({ZIP.stat().st_size / 1e6:.1f} MB)")

rows = []
for m in MODELS:
    if m["tag"] in FAILED:
        continue
    for cond in CONDITIONS:
        p = RAW / m["tag"] / f"{cond}.jsonl"
        got = [json.loads(l) for l in open(p, encoding="utf-8")]
        assert [r["query_id"] for r in got] == [q["query_id"] for q in QUERIES], f"{p}: thiếu hoặc sai thứ tự"
        for r in got:
            q = QBYID[r["query_id"]]
            s = score_v13(q, r["text"], TACTIC_ALIAS)
            pred, gold, other = set(s["pred"]), set(q["gold"]), set(q["other_gold"])
            rows.append(dict(model=m["tag"], condition=cond, kind=CONDITIONS[cond]["kind"], query_id=q["query_id"],
                             base_id=q["base_id"], qform="ver", item_id=q["item_id"], lineage=q["lineage"],
                             revoked=REVOKED[q["query_id"]], template=q["template"], group=q["group"],
                             answer=r["text"], finish_reason=r["finish_reason"], pred="|".join(s["pred"]),
                             gold="|".join(q["gold"]), other_gold="|".join(q["other_gold"]), em=s["em"],
                             outcome=s["outcome"], drift_ext=s["drift_ext"], label_drift=s["label_drift"],
                             inconsistent=s["inconsistent"], outcome13=s["outcome13"], stale13=s["stale13"],
                             format=s["format"], no_answer=s["outcome"] == "NO_ANSWER",
                             stale_union=s["drift_ext"] and bool(pred & (gold - other)),
                             stale_old=s["drift_ext"] and not (pred & (gold - other)),
                             conflict=conflict_flag(r["text"]) if CONDITIONS[cond].get("conflict") else None))
if not rows:
    (OUT / "tech_fail.json").write_text(json.dumps(FAILED, indent=2, ensure_ascii=False))
    pack()
    raise RuntimeError("Mọi mô hình lỗi kỹ thuật ở nhánh này (xem tech_fail.json). Notebook đã đóng gói gui_thay.zip "
                       "kèm log — em gửi thầy file đó, không cần chạy các ô sau.")
SC = pd.DataFrame(rows)
SC.to_csv(OUT / "scored.csv", index=False)
chk = SC.groupby(["model", "condition"]).agg(n=("query_id", "count"), json=("format", lambda x: (x == "json").mean()),
                                             truncated=("finish_reason", lambda x: (x == "length").sum()))
print("Tỉ lệ JSON thấp nhất:", round(chk.json.min(), 3), "| số câu bị cắt:", int(chk.truncated.sum()))
_json_rate = SC.groupby("model")["format"].apply(lambda x: (x == "json").mean()).to_dict()
_cf_rate = SC[SC.condition.str.startswith("cf_")].groupby("model")["conflict"].apply(lambda x: x.notna().mean()).to_dict()
for m in MODELS:
    t = m["tag"]
    if t not in FAILED and _json_rate.get(t, 0.0) < JSON_MIN:
        FAILED[t] = f"chạy chính: tỉ lệ JSON {_json_rate.get(t, 0.0):.2f} < {JSON_MIN}"
CF_BAD = {t: round(r, 3) for t, r in _cf_rate.items() if r < JSON_MIN and t not in FAILED}   # chỉ loại khỏi E5
(OUT / "tech_fail.json").write_text(json.dumps(dict(FAILED, **{f"{t} (chỉ E5)": f"trường conflict {r}" for t, r in CF_BAD.items()}),
                                               indent=2, ensure_ascii=False))
if FAILED:
    print("⚠ Mô hình lỗi kỹ thuật (tính là không đạt mọi giả thuyết):", FAILED)
if CF_BAD:
    print("⚠ Thiếu trường conflict ở ≥ 5% câu cf_* — loại khỏi E5 (không ảnh hưởng E4):", CF_BAD)
# Mỗi mô hình: mọi điều kiện chạy cùng một cấu hình môi trường (đúng cấu hình đã ghi)
for m in MODELS:
    t = m["tag"]
    if t in FAILED:
        continue
    metas = [json.loads((RAW / t / f"meta_{c}.json").read_text()) for c in CONDITIONS]
    eff, R = effective(m, RUNG[t]), ladder(m)[RUNG[t]]
    for mt in metas:
        assert (mt["max_model_len"], mt["gpu_mem"], mt["engine_kwargs"], mt["blas_env"] or "") == \
            (eff["max_model_len"], eff["gpu_mem"], R["engine"], R["env"].get("TORCH_BLAS_PREFER_CUBLASLT", "")), \
            f"{t}: các điều kiện không cùng cấu hình môi trường — gửi thầy"
print("✓ Mỗi mô hình: mọi điều kiện cùng một cấu hình môi trường")
chk

Tỉ lệ JSON thấp nhất: 1.0 | số câu bị cắt: 0
✓ Mỗi mô hình: mọi điều kiện cùng một cấu hình môi trường


n  json  truncated
model     condition                                 
gemma2-9b cf_new_old            361   1.0          0
          cf_old_new            361   1.0          0
          closed                361   1.0          0
          filter                361   1.0          0
          filter_exp            361   1.0          0
...                             ...   ...        ...
qwen3-8b  mixed                 361   1.0          0
          mixed_label           361   1.0          0
          mixed_label_curfirst  361   1.0          0
          new_old               361   1.0          0
          old_new               361   1.0          0

[66 rows x 3 columns]

## 8. Bảng mô tả theo cấu hình

Mỗi ô là tỉ lệ kèm KTC 95% (bootstrap theo cụm dòng dõi, 2.000 lần). Cột:

- `changed: lỗi thời`, `changed: acc`, `changed: rỗng` — nhóm câu có đáp án đổi;
- `thu hồi: acc` — câu changed có mã bị thu hồi; `không thu hồi: lỗi thời` — câu changed còn lại;
- `stable: acc` — đối chứng; `label: tên cũ` — chỉ nhánh A;
- `gộp hai bản / lỗi thời` — trong các câu lỗi thời, tỉ lệ câu có cả đáp án bản mới.

In [14]:
CLUSTER = "lineage"
TAGS_M = [m["tag"] for m in MODELS]
OK_M = [t for t in TAGS_M if t not in FAILED]

def _agg(df, col):
    g = df.groupby(CLUSTER)[col].agg(["sum", "count"])
    return g["sum"].to_numpy(float), g["count"].to_numpy(float), g.index

def boot_rate(df, col, B=2000):
    if len(df) == 0:
        return (float("nan"),) * 3
    s, c, _ = _agg(df, col)
    idx = np.random.default_rng(SEED).integers(0, len(s), size=(B, len(s)))
    r = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (s.sum() / c.sum(), lo, hi)

def boot_paired(df_a, df_b, col, B=BOOT_B, seed_off=0):
    # Hiệu ghép cặp theo cụm: (hiệu, KTC 2,5%, KTC 97,5%, p bootstrap một phía cho hiệu > 0)
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib), "hai nhóm phải có cùng tập cụm"
    assert (ca == cb).all(), "hai nhóm phải có cùng số câu trong mỗi cụm"
    if len(sa) == 0:
        return (float("nan"),) * 4
    idx = np.random.default_rng(SEED + seed_off).integers(0, len(sa), size=(B, len(sa)))
    r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi, (1 + (r <= 0).sum()) / (B + 1))

def signflip_p(df_a, df_b, col, B=BOOT_B):
    # p một phía (hiệu > 0) bằng phép đổi dấu ngẫu nhiên hiệu của từng cụm (ghép cặp) — như notebook 07
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib) and (ca == cb).all()
    if len(sa) == 0:
        return float("nan")
    d, n = sa - sb, ca.sum()
    t_obs = d.sum() / n
    signs = np.random.default_rng(SEED + 1).choice([-1.0, 1.0], size=(B, len(d)))
    t = (signs * d).sum(1) / n
    return (1 + (t >= t_obs - 1e-12).sum()) / (B + 1)

def boot_groups(df_a, df_b, col, B=BOOT_B):
    # Hiệu tỉ lệ giữa hai nhóm câu khác nhau; cụm dòng dõi chung của hai nhóm được lấy mẫu lại cùng nhau
    ga, gb = df_a.groupby(CLUSTER)[col].agg(["sum", "count"]), df_b.groupby(CLUSTER)[col].agg(["sum", "count"])
    if len(ga) == 0 or len(gb) == 0:
        return (float("nan"),) * 3
    keys = sorted(set(ga.index) | set(gb.index))
    ga, gb = ga.reindex(keys, fill_value=0), gb.reindex(keys, fill_value=0)
    sa, ca, sb, cb = (x.to_numpy(float) for x in (ga["sum"], ga["count"], gb["sum"], gb["count"]))
    idx = np.random.default_rng(SEED + 4).integers(0, len(keys), size=(B, len(keys)))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.nanpercentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def fmt_b(t):
    return "—" if t[0] != t[0] else f"{t[0]:.2f} [{t[1]:.2f}, {t[2]:.2f}]"

def sel(model, conds, group=None, filt=None, data=None):
    conds = [conds] if isinstance(conds, str) else conds
    d = SC if data is None else data
    d = d[(d.model == model) & d.condition.isin(conds)]
    if group is not None:
        d = d[d.group == group]
    if filt is not None:
        d = filt(d)
    return d.sort_values(["condition", "base_id"])

rev_only    = lambda d: d[d.revoked]
nonrev_only = lambda d: d[~d.revoked]
GROUPS = [g for g in ("changed", "label", "stable") if (SC.group == g).any()]
T1 = []
for m in OK_M:
    for c in CONDITIONS:
        ch = sel(m, c, "changed")
        st = ch[ch.drift_ext]
        row = {"model": m, "condition": c,
               "changed: lỗi thời": fmt_b(boot_rate(ch, "drift_ext")), "changed: acc": fmt_b(boot_rate(ch, "em")),
               "changed: rỗng": fmt_b(boot_rate(ch, "no_answer")),
               "thu hồi: acc": fmt_b(boot_rate(rev_only(ch), "em")),
               "không thu hồi: lỗi thời": fmt_b(boot_rate(nonrev_only(ch), "drift_ext")),
               "gộp hai bản / lỗi thời": f"{st.stale_union.mean():.2f} ({len(st)})" if len(st) else "—"}
        if "stable" in GROUPS:
            row["stable: acc"] = fmt_b(boot_rate(sel(m, c, "stable"), "em"))
        if "label" in GROUPS:
            row["label: tên cũ"] = fmt_b(boot_rate(sel(m, c, "label"), "label_drift"))
        T1.append(row)
T1 = pd.DataFrame(T1)
T1.to_csv(OUT / "summary_conditions.csv", index=False)
# Bảng gọn để đọc nhanh: tỉ lệ lỗi thời (changed) theo mô hình × cấu hình
display(SC[SC.group == "changed"].pivot_table(index="condition", columns="model", values="drift_ext", aggfunc="mean")
        .reindex(list(CONDITIONS)).round(2))
T1

model,gemma2-9b,llama3.1-8b,mistral7b-v0.3,phi4-mini,qwen3-14b,qwen3-8b
condition,,,,,,
closed,0.64,0.61,0.69,0.71,0.68,0.69
mixed,0.87,0.70,0.91,0.92,0.93,0.95
mixed_label,0.41,0.28,0.41,0.53,0.52,0.52
mixed_label_curfirst,0.42,0.36,0.47,0.68,0.51,0.47
filter,0.08,0.11,0.20,0.37,0.06,0.11
filter_exp,0.02,0.02,0.10,0.33,0.00,0.03
filter_exp_note,0.00,0.04,0.08,0.03,0.00,0.00
old_new,0.75,0.59,0.95,0.94,0.91,0.95
new_old,0.92,0.42,0.94,0.94,0.95,0.97


,model,condition,changed: lỗi thời,changed: acc,changed: rỗng,thu hồi: acc,không thu hồi: lỗi thời,gộp hai bản / lỗi thời,stable: acc
0,qwen3-8b,closed,"0.69 [0.64, 0.73]","0.00 [0.00, 0.02]","0.00 [0.00, 0.00]","0.01 [0.00, 0.02]","0.20 [0.00, 0.50]",0.00 (138),"0.54 [0.49, 0.58]"
1,qwen3-8b,mixed,"0.95 [0.92, 0.98]","0.04 [0.02, 0.08]","0.00 [0.00, 0.00]","0.02 [0.00, 0.04]","0.30 [0.00, 0.60]",0.00 (191),"0.97 [0.94, 1.00]"
2,qwen3-8b,mixed_label,"0.52 [0.44, 0.60]","0.46 [0.38, 0.54]","0.00 [0.00, 0.02]","0.44 [0.36, 0.52]","0.00 [0.00, 0.00]",0.00 (105),"0.97 [0.94, 1.00]"
3,qwen3-8b,mixed_label_curfirst,"0.47 [0.39, 0.55]","0.49 [0.41, 0.57]","0.02 [0.00, 0.04]","0.47 [0.39, 0.55]","0.00 [0.00, 0.00]",0.00 (95),"0.97 [0.95, 0.99]"
4,qwen3-8b,filter,"0.11 [0.08, 0.16]","0.71 [0.63, 0.78]","0.14 [0.09, 0.20]","0.70 [0.62, 0.77]","0.00 [0.00, 0.00]",0.00 (23),"0.99 [0.98, 1.00]"
...,...,...,...,...,...,...,...,...,...
61,qwen3-14b,filter_exp_note,"0.00 [0.00, 0.00]","0.98 [0.95, 1.00]","0.01 [0.00, 0.04]","0.98 [0.96, 1.00]","0.00 [0.00, 0.00]",—,"0.91 [0.87, 0.95]"
62,qwen3-14b,old_new,"0.91 [0.86, 0.94]","0.09 [0.06, 0.14]","0.00 [0.00, 0.00]","0.09 [0.06, 0.14]","0.90 [0.70, 1.00]",0.51 (182),"1.00 [1.00, 1.00]"
63,qwen3-14b,new_old,"0.95 [0.91, 0.97]","0.05 [0.03, 0.09]","0.00 [0.00, 0.00]","0.05 [0.03, 0.09]","0.90 [0.70, 1.00]",0.27 (190),"1.00 [1.00, 1.00]"
64,qwen3-14b,cf_old_new,"0.82 [0.76, 0.87]","0.18 [0.13, 0.24]","0.00 [0.00, 0.00]","0.19 [0.14, 0.24]","0.90 [0.70, 1.00]",0.45 (164),"1.00 [1.00, 1.00]"


## 9. E4: kiểm định phòng thủ đầu-cuối (R1–R3)

Mỗi dòng là một (mô hình × giả thuyết). Hiệu ghép cặp trên cùng câu hỏi; p một phía từ phép đổi dấu theo cụm dòng dõi (10.000 lần); KTC 95% bootstrap theo cụm; Holm trong họ R1–R3 của từng mô hình.

| | Giả thuyết | So sánh (dự đoán > 0) | Câu |
|---|---|---|---|
| **R1** | Lọc theo phiên bản giảm lỗi thời | lỗi thời(`mixed`) − lỗi thời(`filter`) | changed |
| **R2** | Truy hồi theo dòng dõi giúp trả lời đúng mã bị thu hồi | acc(`filter_exp_note`) − acc(`filter`) | changed, có mã bị thu hồi |
| **R3** | Nhãn tin cậy giảm lỗi thời trên truy hồi thật | lỗi thời(`mixed`) − lỗi thời(`mixed_label`) | changed |

**Quy tắc (định trước):**

- R1, R3 **không đánh giá được** ở một mô hình nếu lỗi thời ở `mixed` < 0,15 (sàn); R2 không đánh giá được nếu acc(`filter`) trên câu có mã bị thu hồi > 0,85 (trần). Khi đó giả thuyết bị bỏ khỏi họ Holm và khỏi mẫu số của mô hình đó.
- Mỗi nhánh: giả thuyết **được ủng hộ** nếu đạt (p Holm < 0,05) ở ≥ ⌈0,8 × số mô hình đánh giá được⌉ mô hình **và** không mô hình nào có KTC 95% nằm hẳn dưới 0. Dưới 3 mô hình đánh giá được thì giả thuyết không đánh giá được ở nhánh đó.
- Mô hình lỗi kỹ thuật được tính là không đạt (vẫn nằm trong mẫu số), không bao giờ tính là ngược chiều.

In [15]:
changed_f = lambda d: d[d.group == "changed"]
rev_changed = lambda d: d[(d.group == "changed") & d.revoked]
HYPS = {   # id: (tên, cột, điều kiện phía a, điều kiện phía b, bộ lọc)
    "R1": ("Lọc theo phiên bản giảm lỗi thời", "drift_ext", "mixed", "filter", changed_f),
    "R2": ("Truy hồi theo dòng dõi giúp mã bị thu hồi (acc)", "em", "filter_exp_note", "filter", rev_changed),
    "R3": ("Nhãn tin cậy giảm lỗi thời trên truy hồi thật", "drift_ext", "mixed", "mixed_label", changed_f),
}

def holm(ps):
    order = np.argsort(ps)
    adj, run = np.empty(len(ps)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = run
    return adj

def evaluable(m, hid):
    if m in FAILED:
        return True                                     # lỗi kỹ thuật: vẫn trong mẫu số, tính là không đạt
    if hid in ("R1", "R3"):
        return sel(m, "mixed", filt=changed_f)["drift_ext"].mean() >= FLOOR
    return sel(m, "filter", filt=rev_changed)["em"].mean() <= CEIL

def confirm_table(strict=False):
    out = []
    for m in TAGS_M:
        rows = []
        for hid, (name, col, ca, cb, filt) in HYPS.items():
            col = "stale13" if (strict and col == "drift_ext") else col
            ev = evaluable(m, hid)
            r = {"model": m, "giả thuyết": hid, "tên": name, "chỉ số": col, "đánh giá được": ev}
            if m in FAILED:
                r.update({"hiệu [KTC 95%]": "lỗi kỹ thuật", "p một phía": float("nan"), "n cụm": 0, "_lo": np.nan, "_hi": np.nan})
            else:
                a, b = sel(m, ca, filt=filt), sel(m, cb, filt=filt)
                d = boot_paired(a, b, col)
                r.update({"hiệu [KTC 95%]": fmt_b(d), "p một phía": signflip_p(a, b, col) if ev else float("nan"),
                          "n cụm": a[CLUSTER].nunique(), "_lo": d[1], "_hi": d[2]})
            rows.append(r)
        fam = [r for r in rows if r["đánh giá được"] and m not in FAILED]
        adj = holm(np.array([r["p một phía"] for r in fam])) if fam else []
        for r, x in zip(fam, adj):
            r["p Holm"] = x
            r["đạt"] = "✓" if x < ALPHA else "·"
        for r in rows:
            if "đạt" not in r:
                r["p Holm"] = float("nan")
                r["đạt"] = "lỗi kỹ thuật" if m in FAILED else "không đánh giá được"
            r["ngược chiều"] = bool(r["_hi"] < 0) and m not in FAILED and r["đánh giá được"]
        out += rows
    return pd.DataFrame(out)

def branch_verdicts(T):
    res = []
    for hid in HYPS:
        t = T[(T["giả thuyết"] == hid) & T["đánh giá được"]]
        n_ev, n_pass, n_rev = len(t), int((t["đạt"] == "✓").sum()), int(t["ngược chiều"].sum())
        need = math.ceil(PASS_SHARE * n_ev - 1e-9)
        if n_ev < 3:
            v = "không đánh giá được"
        elif n_pass >= need and n_rev == 0:
            v = "ĐƯỢC ỦNG HỘ"
        else:
            v = "không được ủng hộ"
        res.append({"giả thuyết": hid, "tên": HYPS[hid][0], "đánh giá được": n_ev, "cần đạt": need,
                    "đạt": n_pass, "ngược chiều": n_rev, "kết luận (nhánh " + CFG["branch"] + ")": v})
    return pd.DataFrame(res)

T2 = confirm_table()
T2.drop(columns=["_lo", "_hi"]).to_csv(OUT / "e4_tests.csv", index=False)
V2 = branch_verdicts(T2)
V2.to_csv(OUT / "e4_verdicts.csv", index=False)
display(T2.drop(columns=["_lo", "_hi"]).round({"p một phía": 4, "p Holm": 4}))
V2

,model,giả thuyết,tên,chỉ số,đánh giá được,hiệu [KTC 95%],p một phía,n cụm,p Holm,đạt,ngược chiều
0,qwen3-8b,R1,Lọc theo phiên bản giảm lỗi thời,drift_ext,True,"0.84 [0.79, 0.88]",0.0001,133,0.0003,✓,False
1,qwen3-8b,R2,Truy hồi theo dòng dõi giúp mã bị thu hồi (acc),em,True,"0.30 [0.23, 0.38]",0.0001,123,0.0003,✓,False
2,qwen3-8b,R3,Nhãn tin cậy giảm lỗi thời trên truy hồi thật,drift_ext,True,"0.43 [0.35, 0.51]",0.0001,133,0.0003,✓,False
3,llama3.1-8b,R1,Lọc theo phiên bản giảm lỗi thời,drift_ext,True,"0.58 [0.52, 0.65]",0.0001,133,0.0003,✓,False
4,llama3.1-8b,R2,Truy hồi theo dòng dõi giúp mã bị thu hồi (acc),em,True,"0.21 [0.15, 0.28]",0.0001,123,0.0003,✓,False
5,llama3.1-8b,R3,Nhãn tin cậy giảm lỗi thời trên truy hồi thật,drift_ext,True,"0.41 [0.35, 0.48]",0.0001,133,0.0003,✓,False
6,gemma2-9b,R1,Lọc theo phiên bản giảm lỗi thời,drift_ext,True,"0.79 [0.72, 0.84]",0.0001,133,0.0003,✓,False
7,gemma2-9b,R2,Truy hồi theo dòng dõi giúp mã bị thu hồi (acc),em,True,"0.05 [-0.03, 0.13]",0.1486,123,0.1486,·,False
8,gemma2-9b,R3,Nhãn tin cậy giảm lỗi thời trên truy hồi thật,drift_ext,True,"0.46 [0.38, 0.54]",0.0001,133,0.0003,✓,False
9,phi4-mini,R1,Lọc theo phiên bản giảm lỗi thời,drift_ext,True,"0.55 [0.48, 0.62]",0.0001,133,0.0003,✓,False


,giả thuyết,tên,đánh giá được,cần đạt,đạt,ngược chiều,kết luận (nhánh B)
0,R1,Lọc theo phiên bản giảm lỗi thời,6,5,6,0,ĐƯỢC ỦNG HỘ
1,R2,Truy hồi theo dòng dõi giúp mã bị thu hồi (acc),6,5,5,0,ĐƯỢC ỦNG HỘ
2,R3,Nhãn tin cậy giảm lỗi thời trên truy hồi thật,6,5,6,0,ĐƯỢC ỦNG HỘ


### 9.1. Phân tích độ nhạy: chỉ số `stale13`

In [16]:
T2s = confirm_table(strict=True)
T2s.drop(columns=["_lo", "_hi"]).to_csv(OUT / "e4_tests_strict.csv", index=False)
V2s = branch_verdicts(T2s)
V2s.to_csv(OUT / "e4_verdicts_strict.csv", index=False)
display(T2s[["model", "giả thuyết", "hiệu [KTC 95%]", "p Holm", "đạt"]].round({"p Holm": 4}))
V2s

,model,giả thuyết,hiệu [KTC 95%],p Holm,đạt
0,qwen3-8b,R1,"0.84 [0.79, 0.88]",0.0003,✓
1,qwen3-8b,R2,"0.30 [0.23, 0.38]",0.0003,✓
2,qwen3-8b,R3,"0.43 [0.35, 0.51]",0.0003,✓
3,llama3.1-8b,R1,"0.59 [0.52, 0.65]",0.0003,✓
4,llama3.1-8b,R2,"0.21 [0.15, 0.28]",0.0003,✓
5,llama3.1-8b,R3,"0.41 [0.35, 0.48]",0.0003,✓
6,gemma2-9b,R1,"0.78 [0.72, 0.84]",0.0003,✓
7,gemma2-9b,R2,"0.05 [-0.03, 0.13]",0.1486,·
8,gemma2-9b,R3,"0.46 [0.38, 0.54]",0.0003,✓
9,phi4-mini,R1,"0.56 [0.49, 0.63]",0.0003,✓


,giả thuyết,tên,đánh giá được,cần đạt,đạt,ngược chiều,kết luận (nhánh B)
0,R1,Lọc theo phiên bản giảm lỗi thời,6,5,6,0,ĐƯỢC ỦNG HỘ
1,R2,Truy hồi theo dòng dõi giúp mã bị thu hồi (acc),6,5,5,0,ĐƯỢC ỦNG HỘ
2,R3,Nhãn tin cậy giảm lỗi thời trên truy hồi thật,6,5,6,0,ĐƯỢC ỦNG HỘ


### 9.2. So sánh mô tả (định trước, không có quy tắc kết luận)

Hiệu ghép cặp, KTC 95% hai phía:

- `curfirst − label`: xếp đoạn hiện hành lên trước có giúp thêm so với chỉ gắn nhãn không (lỗi thời, changed);
- `exp − filter`, `exp_note − exp`: phần đóng góp của mở rộng truy vấn và của ghi chú dòng dõi (acc, mã bị thu hồi);
- `full − mixed`: phòng thủ trọn bộ so với kho trộn ngây thơ (lỗi thời và acc, changed);
- `stable: full − mixed`: phòng thủ có làm hại câu đối chứng không (acc, stable);
- `mixed − closed`: RAG ngây thơ so với không RAG (lỗi thời, changed).

In [17]:
DESC = []
COMP = [("curfirst − label (lỗi thời)", "mixed_label_curfirst", "mixed_label", "drift_ext", changed_f),
        ("exp − filter (acc, thu hồi)", "filter_exp", "filter", "em", rev_changed),
        ("exp_note − exp (acc, thu hồi)", "filter_exp_note", "filter_exp", "em", rev_changed),
        ("full − mixed (lỗi thời)", "filter_exp_note", "mixed", "drift_ext", changed_f),
        ("full − mixed (acc)", "filter_exp_note", "mixed", "em", changed_f),
        ("stable: full − mixed (acc)", "filter_exp_note", "mixed", "em", lambda d: d[d.group == "stable"]),
        ("mixed − closed (lỗi thời)", "mixed", "closed", "drift_ext", changed_f)]
for m in OK_M:
    for name, ca, cb, col, filt in COMP:
        DESC.append({"model": m, "so sánh": name, "hiệu [KTC 95%]": fmt_b(boot_paired(sel(m, ca, filt=filt),
                                                                                       sel(m, cb, filt=filt), col, seed_off=3))})
DESC = pd.DataFrame(DESC, columns=["model", "so sánh", "hiệu [KTC 95%]"])
DESC.to_csv(OUT / "e4_descriptive.csv", index=False)
DESC.pivot(index="so sánh", columns="model", values="hiệu [KTC 95%]").reindex([c[0] for c in COMP]) if OK_M else DESC

model,gemma2-9b,llama3.1-8b,mistral7b-v0.3,phi4-mini,qwen3-14b,qwen3-8b
so sánh,,,,,,
curfirst − label (lỗi thời),"0.00 [-0.03, 0.04]","0.07 [0.03, 0.12]","0.06 [0.01, 0.12]","0.15 [0.08, 0.22]","-0.01 [-0.05, 0.03]","-0.05 [-0.09, -0.01]"
"exp − filter (acc, thu hồi)","0.12 [0.06, 0.17]","0.16 [0.11, 0.22]","0.15 [0.09, 0.22]","0.08 [0.01, 0.16]","0.10 [0.06, 0.15]","0.10 [0.04, 0.17]"
"exp_note − exp (acc, thu hồi)","-0.07 [-0.12, -0.01]","0.05 [0.01, 0.09]","0.06 [0.02, 0.11]","0.23 [0.15, 0.32]","0.05 [0.02, 0.10]","0.20 [0.14, 0.27]"
full − mixed (lỗi thời),"-0.87 [-0.92, -0.82]","-0.66 [-0.72, -0.59]","-0.83 [-0.88, -0.77]","-0.89 [-0.93, -0.85]","-0.93 [-0.96, -0.89]","-0.95 [-0.98, -0.92]"
full − mixed (acc),"0.72 [0.65, 0.78]","0.69 [0.63, 0.75]","0.80 [0.75, 0.86]","0.73 [0.67, 0.79]","0.92 [0.87, 0.95]","0.95 [0.92, 0.98]"
stable: full − mixed (acc),"0.02 [0.00, 0.04]","0.10 [0.05, 0.15]","0.01 [-0.02, 0.05]","0.02 [0.01, 0.05]","0.04 [0.00, 0.09]","0.02 [0.00, 0.04]"
mixed − closed (lỗi thời),"0.23 [0.16, 0.29]","0.09 [0.00, 0.17]","0.22 [0.16, 0.27]","0.21 [0.16, 0.26]","0.25 [0.20, 0.31]","0.26 [0.21, 0.31]"


## 10. E5: mô hình có nhận ra ngữ cảnh mâu thuẫn không? (chỉ mô tả)

Ngữ cảnh oracle **không nhãn**, gộp hai thứ tự. Ở nhóm `changed`, đoạn cũ và đoạn mới mâu thuẫn về mã đáp án; ở nhóm `stable`, hai đoạn cho cùng đáp án (báo mâu thuẫn ở đây là báo nhầm). Nhóm `label` (chỉ nhánh A) cùng mã nhưng khác **tên** tactic, nên là mâu thuẫn ở mức tên, không phải báo nhầm.

- `báo mâu thuẫn`: tỉ lệ `"conflict": true`, theo nhóm (câu thiếu trường `conflict` không tính, số câu thiếu ghi ở cột riêng); `changed − stable`: hiệu, KTC 95% hai phía (bootstrap theo cụm dòng dõi, cụm có mặt ở cả hai nhóm được lấy mẫu cùng nhau);
- `lỗi thời: cf − không cf`, `acc: cf − không cf`: cho phép báo mâu thuẫn có đổi câu trả lời không (ghép cặp, KTC 95% hai phía);
- `lỗi thời khi báo / khi không báo`: tỉ lệ lỗi thời trong các câu changed mô hình báo / không báo mâu thuẫn.

In [18]:
CF, BASE_O = ["cf_old_new", "cf_new_old"], ["old_new", "new_old"]
SC["conflict_true"] = SC["conflict"].eq(True)
E5 = []
for m in [t for t in OK_M if t not in CF_BAD]:
    ch_cf, st_cf = sel(m, CF, "changed"), sel(m, CF, "stable")
    ch_b = sel(m, BASE_O, "changed")
    ch_cf_al = ch_cf.assign(condition=ch_cf.condition.str.replace("cf_", "", regex=False)).sort_values(["condition", "base_id"])
    assert (ch_cf_al.base_id.values == ch_b.base_id.values).all() and (ch_cf_al.condition.values == ch_b.condition.values).all()
    nn = lambda d: d[d.conflict.notna()]              # câu thiếu trường conflict không vào tỉ lệ báo
    row = {"model": m, "thiếu trường conflict": int(sel(m, CF).conflict.isna().sum()),
           "báo mâu thuẫn: changed": fmt_b(boot_rate(nn(ch_cf), "conflict_true")),
           "báo mâu thuẫn: stable": fmt_b(boot_rate(nn(st_cf), "conflict_true"))}
    if "label" in GROUPS:
        row["báo mâu thuẫn: label"] = fmt_b(boot_rate(nn(sel(m, CF, "label")), "conflict_true"))
    row["changed − stable"] = fmt_b(boot_groups(ch_cf[ch_cf.conflict.notna()], st_cf[st_cf.conflict.notna()], "conflict_true"))
    row["lỗi thời: không cf"] = f"{ch_b.drift_ext.mean():.2f}"
    row["lỗi thời: cf − không cf"] = fmt_b(boot_paired(ch_cf_al, ch_b, "drift_ext", seed_off=5))
    row["acc: cf − không cf"] = fmt_b(boot_paired(ch_cf_al, ch_b, "em", seed_off=6))
    fl = ch_cf[ch_cf.conflict.eq(True)]
    nf = ch_cf[ch_cf.conflict.eq(False)]
    row["lỗi thời khi báo / khi không báo"] = (f"{fl.drift_ext.mean():.2f} ({len(fl)}) / " if len(fl) else "— (0) / ") + \
                                              (f"{nf.drift_ext.mean():.2f} ({len(nf)})" if len(nf) else "— (0)")
    row["gộp hai bản / lỗi thời: không cf → cf"] = (f"{ch_b[ch_b.drift_ext].stale_union.mean():.2f} → "
                                                    f"{ch_cf[ch_cf.drift_ext].stale_union.mean():.2f}")
    E5.append(row)
E5 = pd.DataFrame(E5)
E5.to_csv(OUT / "e5_conflict.csv", index=False)
E5.T

,0,1,2,3,4,5
model,qwen3-8b,llama3.1-8b,gemma2-9b,phi4-mini,mistral7b-v0.3,qwen3-14b
thiếu trường conflict,0,0,0,0,0,0
báo mâu thuẫn: changed,"0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.03 [0.02, 0.05]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.04 [0.02, 0.06]"
báo mâu thuẫn: stable,"0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.07 [0.04, 0.11]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]"
changed − stable,"0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","-0.04 [-0.08, -0.00]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.04 [0.02, 0.06]"
lỗi thời: không cf,0.96,0.50,0.84,0.94,0.94,0.93
lỗi thời: cf − không cf,"-0.01 [-0.03, 0.01]","0.02 [-0.01, 0.06]","0.01 [-0.03, 0.04]","0.05 [0.03, 0.08]","-0.02 [-0.04, 0.00]","-0.05 [-0.08, -0.02]"
acc: cf − không cf,"0.01 [-0.01, 0.03]","0.06 [0.01, 0.10]","-0.06 [-0.08, -0.03]","-0.05 [-0.08, -0.03]","0.01 [-0.00, 0.03]","0.05 [0.02, 0.07]"
lỗi thời khi báo / khi không báo,— (0) / 0.95 (402),— (0) / 0.52 (402),0.00 (12) / 0.87 (390),— (0) / 0.99 (402),— (0) / 0.92 (402),0.12 (17) / 0.91 (385)
gộp hai bản / lỗi thời: không cf → cf,0.00 → 0.00,0.18 → 0.09,0.00 → 0.00,0.00 → 0.00,0.09 → 0.02,0.38 → 0.32


## 11. Độ lặp lại so với lần chạy trước (mô tả)

Prompt `old_new`, `new_old` trùng từng byte với notebook 07, cùng mô hình, cùng mã commit. Bảng so câu trả lời của lần chạy này với lần trước (notebook 07; Qwen3-14B: notebook 08; nhánh A, Qwen3-8B và Llama-3.1-8B: notebook 06). Khác biệt đến từ tính không tất định khi GPU xử lý theo lô: thành phần lô khác nhau giữa hai lần chạy (notebook 06 chạy cả câu `nover` trong cùng điều kiện, nên lô của nó khác nhiều hơn), và cấu hình môi trường nếu khác.

In [19]:
def load_scored(p):
    d = pd.read_csv(p, dtype={"pred": str})
    for c in ("drift_ext", "em"):
        if c in d:
            d[c] = d[c].astype(str).str.strip().str.lower().eq("true")
    if "qform" not in d:
        d["qform"] = "ver"
    d["pred"] = d["pred"].fillna("")
    return d

SRC = [CONF07 / "scored.csv", EXT08 / "scored.csv"]
if CFG["branch"] == "A":
    SRC.append(ROOT / "results" / "mech-18.1-19.2" / "scored.csv")
PREV = {p: load_scored(p) for p in SRC if p.exists()}
REP = []
for m in OK_M:
    for p, d in PREV.items():
        old = d[(d.model == m) & d.condition.isin(BASE_O) & (d.qform == "ver")]
        if len(old) == 0:
            continue
        new = SC[(SC.model == m) & SC.condition.isin(BASE_O)].copy()
        new["pred"] = new["pred"].fillna("")
        j = new.merge(old, on=["query_id", "condition"], suffixes=("", "_prev"))
        assert len(j) == len(new), f"{m}: lần chạy trước thiếu câu ({p})"
        REP.append({"model": m, "lần trước": str(p.parent.relative_to(ROOT)), "n": len(j),
                    "trùng văn bản": round((j.answer == j.answer_prev).mean(), 3) if "answer_prev" in j else float("nan"),
                    "trùng đáp án": round((j.pred == j.pred_prev).mean(), 3),
                    "lỗi thời (changed): lần này / lần trước":
                        f"{j[j.group == 'changed'].drift_ext.mean():.3f} / {j[j.group == 'changed'].drift_ext_prev.mean():.3f}"})
        break
REP = pd.DataFrame(REP)
REP.to_csv(OUT / "reproducibility.csv", index=False)
REP

,model,lần trước,n,trùng văn bản,trùng đáp án,lỗi thời (changed): lần này / lần trước
0,qwen3-8b,results/conf-6.3-7.2,722,1.000,1.000,0.960 / 0.960
1,llama3.1-8b,results/conf-6.3-7.2,722,0.994,0.996,0.502 / 0.502
2,gemma2-9b,results/conf-6.3-7.2,722,0.996,0.996,0.836 / 0.833
3,phi4-mini,results/conf-6.3-7.2,722,1.000,1.000,0.935 / 0.935
4,mistral7b-v0.3,results/conf-6.3-7.2,722,1.000,1.000,0.943 / 0.943
5,qwen3-14b,results/ext14b-6.3-7.2,722,1.000,1.000,0.925 / 0.925


## 12. Ghi manifest và đóng gói gửi thầy

In [20]:
pack()

Đã tạo results/e2e-6.3-7.2/gui_thay.zip (4.3 MB)


## 13. Gửi thầy

Sau **mỗi** lần chạy (nhánh A, rồi nhánh B), em gửi thầy hai thứ:

- notebook đã chạy;
- file `results/e2e-<lần chuyển>/gui_thay.zip`.

Nếu một ô báo lỗi, em dừng lại và gửi thầy notebook cùng file log trong `results/e2e-<lần chuyển>/logs/`.